# 한국은행 KOSIS 데이터 EDA

로컬 `data/kosis/hankook_tables.json`과 `data/tables/*.csv`를 분석합니다. **검색기와 골든셋 설계**에 필요한 표별 항목, 분류 차원, 단위, 수록 범위와 품질 점검에 초점을 둡니다. API를 호출하거나 원본 파일을 수정하지 않습니다.

아래 셀을 위에서부터 실행한 뒤, **모든 출력이 포함된 `.ipynb`를 저장**해 전달해 주세요. 그 실행 결과를 확인한 후 EDA Markdown 보고서를 작성합니다.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from IPython.display import Markdown, display
from statistics import median
import csv
import json
import re

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "data/kosis/hankook_tables.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("프로젝트 폴더 또는 그 하위 폴더에서 노트북을 실행하세요.")
CATALOG = ROOT / "data/kosis/hankook_tables.json"
TABLE_DIR = ROOT / "data/tables"

def show_table(headers, rows):
    def clean(value):
        return str(value).replace("|", r"\|").replace("\n", " ")
    lines = ["| " + " | ".join(map(clean, headers)) + " |",
             "|" + "|".join("---" for _ in headers) + "|"]
    lines += ["| " + " | ".join(map(clean, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

print("프로젝트 폴더:", ROOT)


프로젝트 폴더: C:\Users\cheon\StatBridge


In [2]:
catalog = json.loads(CATALOG.read_text(encoding="utf-8"))
tables = catalog["tables"]
catalog_ids = [t["tbl_id"] for t in tables]
csv_files = {p.stem: p for p in TABLE_DIR.glob("*.csv")}
missing_ids = sorted(set(catalog_ids) - set(csv_files))
extra_ids = sorted(set(csv_files) - set(catalog_ids))
names = {t["tbl_id"]: t["tbl_nm"] for t in tables}
categories = {t["tbl_id"]: t["path"][0] if t.get("path") else "미분류" for t in tables}

print("카탈로그 표:", len(tables), "/ 고유 표 ID:", len(set(catalog_ids)))
print("CSV 파일:", len(csv_files), "/ CSV가 없는 표:", missing_ids)
print("카탈로그에 없는 CSV:", extra_ids)
print("카탈로그 수집일:", catalog.get("collected_at"))


카탈로그 표: 349 / 고유 표 ID: 349
CSV 파일: 347 / CSV가 없는 표: ['DT_284Y001', 'DT_284Y002']
카탈로그에 없는 CSV: []
카탈로그 수집일: 2026-09-14T06:26:16.468657+00:00


## CSV 전체 스캔

347개 CSV를 **파일별로 한 번씩** 읽고 각 파일의 행만 집계합니다. 전체 데이터를 한꺼번에 메모리에 적재하지 않습니다. 단, 중복 키 확인을 위해 현재 파일의 키 집합은 메모리에 보관합니다. 수록 시점(`PRD_DE`)은 주기별 원본 코드의 최소·최대로 표시하며 날짜로 임의 변환하지 않습니다.

중복 후보의 키는 `PRD_SE + PRD_DE + ITM_ID + 파일에 존재하는 C1~C4`입니다. 같은 키가 실제 오류인지 여부는 원본 통계표와 대조해야 합니다.


In [3]:
table_summary = []
header_variants = Counter()
period_rows = Counter()
non_numeric_examples = Counter()

for index, (table_id, path) in enumerate(sorted(csv_files.items()), 1):
    rows = blank_value = non_numeric = blank_period = blank_item = 0
    id_mismatch = duplicate_keys = blank_unit = blank_update = 0
    items, units = set(), set()
    period_points = defaultdict(set)
    periods = Counter()
    last_update = ""
    with path.open(encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        headers = tuple(reader.fieldnames or ())
        header_variants[headers] += 1
        dimensions = [f"C{i}" for i in range(1, 5) if f"C{i}" in headers]
        dimension_values = {column: set() for column in dimensions}
        dimension_labels = {column: set() for column in dimensions}
        blank_dimensions = Counter()
        key_columns = ["PRD_SE", "PRD_DE", "ITM_ID", *dimensions]
        seen_keys = set()
        for row in reader:
            rows += 1
            if row.get("TBL_ID") != table_id:
                id_mismatch += 1
            key = tuple(row.get(column, "") for column in key_columns)
            if key in seen_keys:
                duplicate_keys += 1
            else:
                seen_keys.add(key)
            item = (row.get("ITM_ID") or "").strip()
            if item:
                items.add(item)
            else:
                blank_item += 1
            for column in dimensions:
                value = (row.get(column) or "").strip()
                if value:
                    dimension_values[column].add(value)
                else:
                    blank_dimensions[column] += 1
                label = (row.get(f"{column}_OBJ_NM") or "").strip()
                if label:
                    dimension_labels[column].add(label)
            if "UNIT_NM" in headers:
                unit = (row.get("UNIT_NM") or "").strip()
                if unit:
                    units.add(unit)
                else:
                    blank_unit += 1
            frequency = (row.get("PRD_SE") or "").strip()
            period = (row.get("PRD_DE") or "").strip()
            if frequency and period:
                periods[frequency] += 1
                period_rows[frequency] += 1
                period_points[frequency].add(period)
            else:
                blank_period += 1
            updated = (row.get("LST_CHN_DE") or "").strip()
            if updated:
                last_update = max(last_update, updated)
            else:
                blank_update += 1
            value = (row.get("DT") or "").strip()
            if not value:
                blank_value += 1
            else:
                try:
                    float(value.replace(",", ""))
                except ValueError:
                    non_numeric += 1
                    if value in non_numeric_examples or len(non_numeric_examples) < 20:
                        non_numeric_examples[value] += 1
    table_summary.append({
        "table_id": table_id, "category": categories.get(table_id, "카탈로그 없음"),
        "rows": rows, "item_count": len(items), "blank_item": blank_item,
        "dimensions": dimensions,
        "dimension_counts": {column: len(values) for column, values in dimension_values.items()},
        "dimension_labels": {column: sorted(values) for column, values in dimension_labels.items()},
        "blank_dimensions": dict(blank_dimensions),
        "has_unit_column": "UNIT_NM" in headers, "unit_count": len(units),
        "unit_values": sorted(units), "blank_unit": blank_unit,
        "period_rows": dict(periods),
        "period_counts": {freq: len(points) for freq, points in period_points.items()},
        "period_spans": {freq: (min(points), max(points))
                         for freq, points in period_points.items()},
        "last_update": last_update, "blank_update": blank_update,
        "blank_value": blank_value, "non_numeric": non_numeric,
        "blank_period": blank_period, "id_mismatch": id_mismatch,
        "duplicate_keys": duplicate_keys,
    })
    if index % 50 == 0 or index == len(csv_files):
        print(f"스캔 완료: {index}/{len(csv_files)} 파일")


스캔 완료: 50/347 파일
스캔 완료: 100/347 파일
스캔 완료: 150/347 파일
스캔 완료: 200/347 파일
스캔 완료: 250/347 파일
스캔 완료: 300/347 파일
스캔 완료: 347/347 파일


## 분류·구조·수록 범위

분류별 수집률과 데이터 규모, CSV 열 구조, 표별 조회 조건을 확인합니다. `UNIT_NM` 열 부재는 표의 구조 차이로 표시하며 곧바로 데이터 오류로 판단하지 않습니다.


In [4]:
profile_by_id = {s["table_id"]: s for s in table_summary}
category_counts = Counter(categories.values())
category_profiles = defaultdict(list)
for profile in table_summary:
    category_profiles[profile["category"]].append(profile)

print("총 데이터 행:", f'{sum(s["rows"] for s in table_summary):,}')
print("분류별 표·CSV·행 수·표당 중앙 행 수:")
show_table(["분류", "카탈로그 표", "CSV", "데이터 행", "표당 중앙 행"], [
    (category, count, len(category_profiles[category]),
     f'{sum(s["rows"] for s in category_profiles[category]):,}',
     f'{median(s["rows"] for s in category_profiles[category]):,.0f}'
     if category_profiles[category] else "—")
    for category, count in category_counts.most_common()
])
print("주기별 데이터 행:")
show_table(["PRD_SE", "행 수"], [(key, f"{count:,}") for key, count in period_rows.most_common()])
print("분류 차원 수·단위 열 보유 여부별 CSV 수:")
shape_counts = Counter((len(s["dimensions"]), s["has_unit_column"]) for s in table_summary)
show_table(["분류 차원 수", "UNIT_NM 열", "CSV 수"], [
    (dimension_count, "있음" if has_unit else "없음", count)
    for (dimension_count, has_unit), count in sorted(shape_counts.items())
])
print("헤더 유형:", len(header_variants))
show_table(["열 수", "파일 수", "분류 코드 열", "UNIT_NM 열"], [
    (len(columns), count,
     ", ".join(f"C{i}" for i in range(1, 5) if f"C{i}" in columns) or "없음",
     "있음" if "UNIT_NM" in columns else "없음")
    for columns, count in header_variants.most_common()
])


총 데이터 행: 2,145,280
분류별 표·CSV·행 수·표당 중앙 행 수:


| 분류 | 카탈로그 표 | CSV | 데이터 행 | 표당 중앙 행 |
|---|---|---|---|---|
| 통화금융통계 | 117 | 117 | 519,175 | 1,925 |
| 기업경영분석 | 78 | 78 | 241,048 | 16 |
| 국민계정 | 60 | 60 | 305,092 | 3,164 |
| 국제수지통계 | 18 | 18 | 350,422 | 12,340 |
| 기업경기조사 | 15 | 15 | 241,468 | 5,098 |
| 지급결제통계 | 15 | 15 | 118,983 | 6,080 |
| 자금순환표 | 14 | 12 | 32,903 | 854 |
| 수출입물가조사(2020=100) | 12 | 12 | 7,460 | 668 |
| 생산자물가조사(2020=100) | 6 | 6 | 52,516 | 3,660 |
| 소비자동향조사 | 4 | 4 | 215,004 | 46,248 |
| 국제투자대조표(IIP) | 3 | 3 | 31,205 | 3,024 |
| 대외채무및대외채권 | 3 | 3 | 27,432 | 12,319 |
| 금융기관대출행태조사 | 3 | 3 | 2,004 | 701 |
| 경제심리지수 | 1 | 1 | 568 | 568 |

주기별 데이터 행:


| PRD_SE | 행 수 |
|---|---|
| M | 1,365,020 |
| Q | 388,708 |
| A | 388,000 |
| S | 3,552 |

분류 차원 수·단위 열 보유 여부별 CSV 수:


| 분류 차원 수 | UNIT_NM 열 | CSV 수 |
|---|---|---|
| 1 | 없음 | 4 |
| 1 | 있음 | 166 |
| 2 | 없음 | 19 |
| 2 | 있음 | 70 |
| 3 | 없음 | 30 |
| 3 | 있음 | 57 |
| 4 | 있음 | 1 |

헤더 유형: 8


| 열 수 | 파일 수 | 분류 코드 열 | UNIT_NM 열 |
|---|---|---|---|
| 17 | 165 | C1 | 있음 |
| 22 | 70 | C1, C2 | 있음 |
| 27 | 57 | C1, C2, C3 | 있음 |
| 25 | 30 | C1, C2, C3 | 없음 |
| 20 | 19 | C1, C2 | 없음 |
| 15 | 4 | C1 | 없음 |
| 13 | 1 | C1 | 있음 |
| 32 | 1 | C1, C2, C3, C4 | 있음 |

In [5]:
print("표별 상세 프로필 (CSV가 없는 표 포함):")
show_table(["표 ID", "표 이름", "분류", "행 수", "항목 수", "차원별 값 수",
            "차원명", "단위 종류", "단위", "주기별 시점 수", "수록 시점", "최종 수정일"], [
    (table["tbl_id"], table["tbl_nm"], categories[table["tbl_id"]],
     f'{profile["rows"]:,}' if profile else "CSV 없음",
     profile["item_count"] if profile else "—",
     ", ".join(f"{key}:{value}" for key, value in profile["dimension_counts"].items())
     if profile and profile["dimension_counts"] else "—",
     ", ".join(f"{key}:{'/'.join(values)}" for key, values
               in profile["dimension_labels"].items())
     if profile and profile["dimension_labels"] else "—",
     profile["unit_count"] if profile and profile["has_unit_column"] else "—",
     (", ".join(profile["unit_values"]) or "값 없음")
     if profile and profile["has_unit_column"] else ("열 없음" if profile else "—"),
     ", ".join(f"{key}:{value}" for key, value in sorted(profile["period_counts"].items()))
     if profile else "—",
     "; ".join(f"{key} {start}–{end}" for key, (start, end)
               in sorted(profile["period_spans"].items())) if profile else "—",
     profile["last_update"] or "—" if profile else "—")
    for table in tables
    for profile in [profile_by_id.get(table["tbl_id"])]
])


표별 상세 프로필 (CSV가 없는 표 포함):


| 표 ID | 표 이름 | 분류 | 행 수 | 항목 수 | 차원별 값 수 | 차원명 | 단위 종류 | 단위 | 주기별 시점 수 | 수록 시점 | 최종 수정일 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| DT_101Y001 | M2 상품별 구성내역(말잔, 계절조정계열) | 통화금융통계 | 757 | 1 | C1:11 | C1:계정항목별 | 1 | 십억원 | M:417 | M 197001–200409 | 2025-12-30 |
| DT_101Y002 | M2 상품별 구성내역(말잔, 원계열) | 통화금융통계 | 6,131 | 1 | C1:16 | C1:계정항목별 | 1 | 십억원 | M:537 | M 196001–200409 | 2025-12-30 |
| DT_101Y003 | M2 상품별 구성내역(평잔, 계절조정계열) | 통화금융통계 | 565 | 1 | C1:11 | C1:계정항목별 | 1 | 십억원 | M:225 | M 198601–200409 | 2025-12-30 |
| DT_101Y004 | M2 상품별 구성내역(평잔, 원계열) | 통화금융통계 | 3,472 | 1 | C1:16 | C1:계정항목별 | 1 | 십억원 | M:225 | M 198601–200409 | 2025-12-30 |
| DT_101Y005 | M2 경제주체별 보유현황(말잔, 계절조정계열) | 통화금융통계 | 170 | 1 | C1:5 | C1:계정항목별 | 1 | 십억원 | M:34 | M 200112–200409 | 2025-12-30 |
| DT_101Y006 | M2 경제주체별 보유현황(평잔, 계절조정계열) | 통화금융통계 | 170 | 1 | C1:5 | C1:계정항목별 | 1 | 십억원 | M:34 | M 200112–200409 | 2025-12-30 |
| DT_101Y007 | M2 기관별구성내역(말잔) | 통화금융통계 | 5,417 | 1 | C1:13 | C1:계정항목별 | 1 | 십억원 | M:537 | M 196001–200409 | 2025-12-30 |
| DT_101Y008 | M2 기관별구성내역(평잔) | 통화금융통계 | 2,733 | 1 | C1:13 | C1:계정항목별 | 1 | 십억원 | M:225 | M 198601–200409 | 2025-12-30 |
| DT_101Y009 | 예금취급기관 개관표(말잔) | 통화금융통계 | 2,686 | 1 | C1:79 | C1:계정항목별 | 1 | 십억원 | M:34 | M 200112–200409 | 2025-12-30 |
| DT_101Y010 | 중앙은행 개관표(말잔) | 통화금융통계 | 2,482 | 1 | C1:73 | C1:계정항목별 | 1 | 십억원 | M:34 | M 200112–200409 | 2025-12-30 |
| DT_101Y011 | 기타예금취급기관 개관표(말잔) | 통화금융통계 | 2,856 | 1 | C1:84 | C1:계정항목별 | 1 | 십억원 | M:34 | M 200112–200409 | 2025-12-30 |
| DT_101Y014 | M2 경제주체별 보유현황(말잔, 원계열) | 통화금융통계 | 170 | 1 | C1:5 | C1:계정항목별 | 1 | 십억원 | M:34 | M 200112–200409 | 2025-12-30 |
| DT_101Y015 | M2 경제주체별 보유현황(평잔, 원계열) | 통화금융통계 | 170 | 1 | C1:5 | C1:계정항목별 | 1 | 십억원 | M:34 | M 200112–200409 | 2025-12-30 |
| DT_101Y016 | M1 상품별 구성내역(말잔, 계절조정계열) | 통화금융통계 | 519 | 1 | C1:4 | C1:계정항목별 | 1 | 십억원 | M:417 | M 197001–200409 | 2025-12-30 |
| DT_101Y017 | M1 상품별 구성내역(말잔, 원계열) | 통화금융통계 | 1,925 | 1 | C1:4 | C1:계정항목별 | 1 | 십억원 | M:537 | M 196001–200409 | 2025-12-30 |
| DT_101Y018 | M1 상품별 구성내역(평잔, 계절조정계열) | 통화금융통계 | 519 | 1 | C1:4 | C1:계정항목별 | 1 | 십억원 | M:417 | M 197001–200409 | 2025-12-30 |
| DT_101Y019 | M1 상품별 구성내역(평잔, 원계열) | 통화금융통계 | 1,092 | 1 | C1:4 | C1:계정항목별 | 1 | 십억원 | M:417 | M 197001–200409 | 2025-12-30 |
| DT_102Y001 | 본원통화 구성내역(말잔, 원계열) | 통화금융통계 | 823 | 1 | C1:3 | C1:계정항목별 | 1 | 십억원 | M:275 | M 200310–202608 | 2026-09-20 |
| DT_102Y002 | 본원통화 구성내역(평잔, 원계열) | 통화금융통계 | 823 | 1 | C1:3 | C1:계정항목별 | 1 | 십억원 | M:275 | M 200310–202608 | 2026-09-20 |
| DT_102Y003 | 본원통화 구성내역(말잔, 계절조정계열) | 통화금융통계 | 822 | 1 | C1:3 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_102Y004 | 본원통화 구성내역(평잔, 계절조정계열) | 통화금융통계 | 822 | 1 | C1:3 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_102Y101 | 본원통화 구성내역(말잔, 원계열) | 통화금융통계 | 603 | 1 | C1:3 | C1:계정항목별 | 1 | 십억원 | M:537 | M 196001–200409 | 2025-12-30 |
| DT_102Y102 | 본원통화 구성내역(평잔, 원계열) | 통화금융통계 | 471 | 1 | C1:3 | C1:계정항목별 | 1 | 십억원 | M:405 | M 197101–200409 | 2025-12-30 |
| DT_102Y103 | 본원통화 구성내역(말잔, 계절조정계열) | 통화금융통계 | 102 | 1 | C1:3 | C1:계정항목별 | 1 | 십억원 | M:34 | M 200112–200409 | 2025-12-30 |
| DT_102Y104 | 본원통화 구성내역(평잔, 계절조정계열) | 통화금융통계 | 102 | 1 | C1:3 | C1:계정항목별 | 1 | 십억원 | M:34 | M 200112–200409 | 2025-12-30 |
| DT_103Y002 | 한국은행 주요계정(말잔) | 통화금융통계 | 24,634 | 1 | C1:37 | C1:계정항목별 | 1 | 십억원 | M:679 | M 197001–202607 | 2026-09-20 |
| DT_104Y008 | 예금은행 원화예금(종별, 평잔) | 통화금융통계 | 11,430 | 1 | C1:22 | C1:계정항목별 | 1 | 십억원 | M:667 | M 197101–202607 | 2026-09-20 |
| DT_104Y009 | 예금은행 원화예금(예금주별,말잔) | 통화금융통계 | 7,428 | 1 | C1:12 | C1:계정항목별 | 1 | 십억원 | M:619 | M 197501–202607 | 2026-09-20 |
| DT_104Y010 | 예금은행 정기예금(기간별, 말잔) | 통화금융통계 | 1,770 | 1 | C1:6 | C1:계정항목별 | 1 | 십억원 | M:295 | M 200201–202607 | 2026-09-20 |
| DT_104Y013 | 예금은행 총수신(말잔) | 통화금융통계 | 3,663 | 1 | C1:9 | C1:계정항목별 | 1 | 십억원 | M:407 | M 199209–202607 | 2026-09-20 |
| DT_104Y014 | 예금은행 총수신(평잔) | 통화금융통계 | 3,663 | 1 | C1:9 | C1:계정항목별 | 1 | 십억원 | M:407 | M 199209–202607 | 2026-09-20 |
| DT_104Y015 | 예금은행 원화예금(종별, 말잔) | 통화금융통계 | 12,987 | 1 | C1:22 | C1:계정항목별 | 1 | 십억원 | M:799 | M 196001–202607 | 2026-09-20 |
| DT_104Y016 | 예금은행 대출금(말잔) | 통화금융통계 | 26,306 | 1 | C1:44 | C1:계정항목별 | 1 | 십억원 | M:799 | M 196001–202607 | 2026-09-20 |
| DT_104Y017 | 예금은행 예금회전율 | 통화금융통계 | 5,489 | 1 | C1:11 | C1:계정항목별 | 1 | 회/월 | M:499 | M 198501–202607 | 2026-09-20 |
| DT_105Y001 | 예금규모별 계좌수 및 금액 | 통화금융통계 | 3,552 | 1 | C1:9, C2:2, C3:5 | C1:계정항목별, C2:구분코드별, C3:금액구분코드별 | — | 열 없음 | S:48 | S 200201–202502 | 2026-09-20 |
| DT_111Y001 | Lf 상품별 구성내역(말잔, 계절조정계열) | 통화금융통계 | 518 | 1 | C1:4 | C1:계정항목별 | 1 | 십억원 | M:225 | M 198601–200409 | 2025-12-30 |
| DT_111Y002 | Lf 상품별 구성내역(말잔, 원계열) | 통화금융통계 | 2,277 | 1 | C1:13 | C1:계정항목별 | 1 | 십억원 | M:297 | M 198001–200409 | 2025-12-30 |
| DT_111Y003 | Lf 상품별 구성내역(평잔, 계절조정계열) | 통화금융통계 | 518 | 1 | C1:4 | C1:계정항목별 | 1 | 십억원 | M:225 | M 198601–200409 | 2025-12-30 |
| DT_111Y004 | Lf 상품별 구성내역(평잔, 원계열) | 통화금융통계 | 2,205 | 1 | C1:13 | C1:계정항목별 | 1 | 십억원 | M:225 | M 198601–200409 | 2025-12-30 |
| DT_111Y007 | 비은행금융기관 수신(말잔) | 통화금융통계 | 3,940 | 1 | C1:10 | C1:계정항목별 | 1 | 십억원 | M:394 | M 199310–202607 | 2026-09-20 |
| DT_111Y008 | 비은행금융기관 수신(평잔) | 통화금융통계 | 3,940 | 1 | C1:10 | C1:계정항목별 | 1 | 십억원 | M:394 | M 199310–202607 | 2026-09-20 |
| DT_111Y009 | 비은행금융기관 여신(말잔) | 통화금융통계 | 3,950 | 1 | C1:10 | C1:계정항목별 | 1 | 십억원 | M:395 | M 199309–202607 | 2026-09-20 |
| DT_112Y001 | L(광의유동성) 구성내역(말잔, 계절조정계열) | 통화금융통계 | 170 | 1 | C1:5 | C1:계정항목별 | 1 | 십억원 | M:34 | M 200112–200409 | 2025-12-30 |
| DT_112Y002 | L(광의유동성) 구성내역(말잔, 원계열) | 통화금융통계 | 1,170 | 1 | C1:10 | C1:계정항목별 | 1 | 십억원 | M:117 | M 199501–200409 | 2025-12-30 |
| DT_121Y002 | 예금은행 수신금리(신규취급액 기준) | 통화금융통계 | 9,412 | 1 | C1:27 | C1:계정항목별 | 2 | 연%, 연리% | M:367 | M 199601–202607 | 2026-09-20 |
| DT_121Y004 | 비은행금융기관 수신금리(신규취급액 기준) | 통화금융통계 | 3,313 | 1 | C1:13 | C1:계정항목별 | 1 | 연리% | M:348 | M 199708–202607 | 2026-09-20 |
| DT_121Y006 | 예금은행 대출금리(신규취급액 기준) | 통화금융통계 | 5,855 | 1 | C1:19 | C1:계정항목별 | 2 | 연%, 연리% | M:367 | M 199601–202607 | 2026-09-20 |
| DT_121Y007 | 비은행금융기관 대출금리(신규취급액 기준) | 통화금융통계 | 6,224 | 1 | C1:27 | C1:계정항목별 | 1 | 연리% | M:415 | M 199201–202607 | 2026-09-20 |
| DT_121Y008 | 예금은행 금리수준별 여수신 비중(신규취급액 기준) | 통화금융통계 | 19,856 | 1 | C1:80 | C1:계정항목별 | 1 | % | M:331 | M 199703–202607 | 2026-09-20 |
| DT_121Y010 | 예금은행 고정 및 변동금리대출 비중(신규취급액 기준) | 통화금융통계 | 4,566 | 1 | C1:18 | C1:계정항목별 | 1 | % | M:299 | M 200109–202607 | 2026-09-20 |
| DT_121Y011 | 예금은행 고정 및 변동금리대출 비중(잔액 기준) | 통화금융통계 | 3,378 | 1 | C1:18 | C1:계정항목별 | 1 | % | M:200 | M 200912–202607 | 2026-09-20 |
| DT_121Y013 | 예금은행 수신금리(잔액 기준) | 통화금융통계 | 5,841 | 1 | C1:19 | C1:계정항목별 | 1 | 연리% | M:367 | M 199601–202607 | 2026-09-20 |
| DT_121Y015 | 예금은행 대출금리(잔액 기준) | 통화금융통계 | 4,837 | 1 | C1:19 | C1:계정항목별 | 1 | 연리% | M:367 | M 199601–202607 | 2026-09-20 |
| DT_131Y010 | 산업별대출금(예금은행, 기업형태별) | 통화금융통계 | 1,240 | 1 | C1:20, C2:2 | C1:계정항목별, C2:분류체계별 | 1 | 십억원 | Q:31 | Q 201804–202602 | 2026-09-20 |
| DT_131Y011 | 산업별대출금(예금은행, 용도별) | 통화금융통계 | 3,894 | 1 | C1:21, C2:2 | C1:계정항목별, C2:자금구분별 | 1 | 십억원 | Q:99 | Q 200104–202602 | 2026-09-20 |
| DT_131Y013 | 산업별대출금(예금은행, 전산업) | 통화금융통계 | 4,061 | 1 | C1:43 | C1:계정항목별 | 1 | 십억원 | Q:99 | Q 200104–202602 | 2026-09-20 |
| DT_131Y014 | 산업별대출금(비은행예금취급기관, 전산업) | 통화금융통계 | 3,061 | 1 | C1:43 | C1:계정항목별 | 1 | 십억원 | Q:74 | Q 200801–202602 | 2026-09-20 |
| DT_131Y015 | 산업별대출금(비은행예금취급기관, 용도별) | 통화금융통계 | 2,944 | 1 | C1:21, C2:2 | C1:계정항목별, C2:자금구분별 | 1 | 십억원 | Q:74 | Q 200801–202602 | 2026-09-20 |
| DT_131Y016 | 산업별대출금(예금취급기관, 전산업) | 통화금융통계 | 3,061 | 1 | C1:43 | C1:계정항목별 | 1 | 십억원 | Q:74 | Q 200801–202602 | 2026-09-20 |
| DT_131Y017 | 산업별대출금(예금취급기관, 용도별) | 통화금융통계 | 2,944 | 1 | C1:21, C2:2 | C1:계정항목별, C2:자금구분별 | 1 | 십억원 | Q:74 | Q 200801–202602 | 2026-09-20 |
| DT_131Y018 | 산업별대출금(예금은행, 기업규모별) | 통화금융통계 | 2,200 | 1 | C1:20, C2:5 | C1:계정항목별, C2:기업규모별 | 1 | 십억원 | Q:22 | Q 202101–202602 | 2026-09-20 |
| DT_132Y001 | 산업별대출금(예금은행, 지역별, 전산업) | 통화금융통계 | 76 | 1 | C1:1, C2:1 | C1:계정항목별, C2:지역코드별 | 1 | 십억원 | Q:76 | Q 200703–202602 | 2026-09-20 |
| DT_132Y003 | 산업별대출금(예금은행, 지역별, 용도별) | 통화금융통계 | 76 | 1 | C1:1, C2:1, C3:1 | C1:계정항목별, C2:지역코드별, C3:자금구분별 | 1 | 십억원 | Q:76 | Q 200703–202602 | 2026-09-20 |
| DT_141Y002 | 예금은행 지역별 수신(말잔) | 통화금융통계 | 367 | 1 | C1:1, C2:1, C3:1 | C1:계정항목별, C2:지역코드별, C3:보고기관별 | 1 | 십억원 | M:367 | M 199512–202606 | 2026-09-20 |
| DT_141Y003 | 예금은행 지역별 대출금(말잔) | 통화금융통계 | 367 | 1 | C1:1, C2:1, C3:1 | C1:계정항목별, C2:지역코드별, C3:보고기관별 | 1 | 십억원 | M:367 | M 199512–202606 | 2026-09-20 |
| DT_141Y004 | 예금은행 지역별 기업대출(말잔) | 통화금융통계 | 12,432 | 1 | C1:3, C2:18, C3:1 | C1:계정항목별, C2:지역코드별, C3:보고기관별 | 1 | 십억원 | M:234 | M 200701–202606 | 2026-09-20 |
| DT_141Y006 | 비은행금융기관 지역별 수신(말잔) | 통화금융통계 | 262 | 1 | C1:1, C2:1 | C1:보고기관별, C2:지역코드별 | 1 | 십억원 | M:262 | M 200409–202606 | 2026-09-20 |
| DT_141Y007 | 비은행금융기관 지역별 여신(말잔) | 통화금융통계 | 36,954 | 1 | C1:8, C2:18 | C1:보고기관별, C2:지역코드별 | 1 | 십억원 | M:262 | M 200409–202606 | 2026-09-20 |
| DT_141Y008 | 생명보험회사 지역별 수신(말잔) | 통화금융통계 | 4,604 | 1 | C1:1, C2:18 | C1:보고기관별, C2:지역코드별 | 1 | 십억원 | M:261 | M 200409–202605 | 2026-09-20 |
| DT_141Y009 | 생명보험회사 지역별 여신(말잔) | 통화금융통계 | 4,604 | 1 | C1:1, C2:18 | C1:보고기관별, C2:지역코드별 | 1 | 십억원 | M:261 | M 200409–202605 | 2026-09-20 |
| DT_141Y010 | 비은행금융기관 지역별 기업대출(말잔) | 통화금융통계 | 34,992 | 1 | C1:2, C2:6, C3:18 | C1:계정항목별, C2:보고기관별, C3:지역코드별 | 1 | 십억원 | M:162 | M 201301–202606 | 2026-09-20 |
| DT_151Y001 | 가계신용(업권별, 분기) | 통화금융통계 | 1,900 | 1 | C1:20 | C1:계정항목별 | 1 | 십억원 | Q:95 | Q 200204–202602 | 2026-09-20 |
| DT_151Y002 | 예금취급기관 가계대출(업권별, 월) | 통화금융통계 | 2,192 | 1 | C1:8 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_151Y003 | 예금은행 지역별 가계대출(말잔) | 통화금융통계 | 13,129 | 1 | C1:3, C2:18 | C1:계정항목별, C2:지역코드별 | 1 | 십억원 | M:273 | M 200310–202606 | 2026-09-20 |
| DT_151Y004 | 가계신용(용도별, 분기) | 통화금융통계 | 1,139 | 1 | C1:15 | C1:계정항목별 | 1 | 십억원 | Q:95 | Q 200204–202602 | 2026-09-20 |
| DT_151Y005 | 예금취급기관 가계대출(용도별, 월) | 통화금융통계 | 1,928 | 1 | C1:9 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_151Y006 | 비은행금융기관 지역별 가계대출(말잔) | 통화금융통계 | 31,656 | 1 | C1:8, C2:18 | C1:계정항목별, C2:지역코드별 | 1 | 십억원 | M:223 | M 200712–202606 | 2026-09-20 |
| DT_161Y001 | M1 상품별 구성내역(평잔, 계절조정계열) | 통화금융통계 | 1,096 | 1 | C1:4 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_161Y002 | M1 상품별 구성내역(평잔, 원계열) | 통화금융통계 | 1,096 | 1 | C1:4 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_161Y003 | M1 상품별 구성내역(말잔, 계절조정계열) | 통화금융통계 | 1,096 | 1 | C1:4 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_161Y004 | M1 상품별 구성내역(말잔, 원계열) | 통화금융통계 | 1,096 | 1 | C1:4 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_161Y005 | M2 상품별 구성내역(평잔, 계절조정계열) | 통화금융통계 | 3,014 | 1 | C1:11 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_161Y006 | M2 상품별 구성내역(평잔, 원계열) | 통화금융통계 | 4,384 | 1 | C1:16 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_161Y007 | M2 상품별 구성내역(말잔, 계절조정계열) | 통화금융통계 | 3,014 | 1 | C1:11 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_161Y008 | M2 상품별 구성내역(말잔, 원계열) | 통화금융통계 | 4,384 | 1 | C1:16 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_161Y009 | M2 경제주체별 보유현황(평잔, 계절조정계열) | 통화금융통계 | 1,562 | 1 | C1:11 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_161Y010 | M2 경제주체별 보유현황(평잔, 원계열) | 통화금융통계 | 1,562 | 1 | C1:11 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_161Y011 | M2 경제주체별 보유현황(말잔, 계절조정계열) | 통화금융통계 | 1,562 | 1 | C1:11 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_161Y012 | M2 경제주체별 보유현황(말잔, 원계열) | 통화금융통계 | 1,562 | 1 | C1:11 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_161Y013 | M2 기관별 구성내역(평잔) | 통화금융통계 | 2,192 | 1 | C1:8 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_161Y014 | M2 기관별 구성내역(말잔) | 통화금융통계 | 2,192 | 1 | C1:8 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_161Y015 | 예금취급기관 개관표(말잔) | 통화금융통계 | 21,728 | 1 | C1:107 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_161Y016 | 중앙은행 개관표(말잔) | 통화금융통계 | 20,511 | 1 | C1:104 | C1:계정항목별 | 1 | 십억원 | M:275 | M 200310–202608 | 2026-09-20 |
| DT_161Y017 | 기타예금취급기관 개관표(말잔) | 통화금융통계 | 23,098 | 1 | C1:112 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_171Y001 | Lf 상품별 구성내역(말잔, 계절조정계열) | 통화금융통계 | 1,370 | 1 | C1:5 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_171Y002 | Lf 상품별 구성내역(말잔, 원계열) | 통화금융통계 | 1,370 | 1 | C1:5 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_171Y003 | Lf 구성내역(평잔, 계절조정계열) | 통화금융통계 | 1,370 | 1 | C1:5 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_171Y004 | Lf 구성내역(평잔, 원계열) | 통화금융통계 | 1,370 | 1 | C1:5 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_172Y001 | L(광의유동성) 구성내역(말잔, 계절조정계열) | 통화금융통계 | 1,370 | 1 | C1:5 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_172Y002 | L(광의유동성) 구성내역(말잔, 원계열) | 통화금융통계 | 1,370 | 1 | C1:5 | C1:계정항목별 | 1 | 십억원 | M:274 | M 200310–202607 | 2026-09-20 |
| DT_181Y001 | 차주당 가계대출 신규취급액 | 통화금융통계 | 1,602 | 1 | C1:30 | C1:계정항목별 | 1 | 십만원 | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_181Y002 | 차주당 가계대출 잔액 | 통화금융통계 | 1,440 | 1 | C1:27 | C1:계정항목별 | 1 | 십만원 | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_181Y003 | 연령별 지역별 | 통화금융통계 | 2,160 | 1 | C1:5, C2:8 | C1:구분코드별, C2:계정항목별 | 1 | 십만원 | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_181Y004 | 연령별 지역별 | 통화금융통계 | 2,160 | 1 | C1:5, C2:8 | C1:구분코드별, C2:계정항목별 | 1 | 십만원 | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_181Y005 | 연령별 업권별 | 통화금융통계 | 810 | 1 | C1:5, C2:3 | C1:구분코드별, C2:계정항목별 | 1 | 십만원 | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_181Y006 | 연령별 업권별 | 통화금융통계 | 810 | 1 | C1:5, C2:3 | C1:구분코드별, C2:계정항목별 | 1 | 십만원 | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_181Y007 | 연령별 상품별 | 통화금융통계 | 1,260 | 1 | C1:5, C2:5 | C1:구분코드별, C2:계정항목별 | 1 | 십만원 | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_181Y008 | 연령별 상품별 | 통화금융통계 | 1,260 | 1 | C1:5, C2:5 | C1:구분코드별, C2:계정항목별 | 1 | 십만원 | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_181Y009 | 상품별 업권별 | 통화금융통계 | 720 | 1 | C1:5, C2:3 | C1:구분코드별, C2:계정항목별 | 1 | 십만원 | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_181Y010 | 상품별 업권별 | 통화금융통계 | 720 | 1 | C1:5, C2:3 | C1:구분코드별, C2:계정항목별 | 1 | 십만원 | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_181Y011 | 차주당 주택담보대출 신규취급액 | 통화금융통계 | 1,350 | 1 | C1:25 | C1:계정항목별 | 1 | 십만원 | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_181Y012 | 차주당 주택담보대출 잔액 | 통화금융통계 | 1,188 | 1 | C1:22 | C1:계정항목별 | 1 | 십만원 | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_181Y013 | 가계대출 신규취급액 비중 | 통화금융통계 | 1,602 | 1 | C1:30 | C1:계정항목별 | 1 | % | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_181Y014 | 주택담보대출 잔액 비중 | 통화금융통계 | 1,188 | 1 | C1:22 | C1:계정항목별 | 1 | % | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_181Y015 | 주택담보대출 신규취급액 비중 | 통화금융통계 | 1,350 | 1 | C1:25 | C1:계정항목별 | 1 | % | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_181Y016 | 가계대출 잔액 비중 | 통화금융통계 | 1,440 | 1 | C1:27 | C1:계정항목별 | 1 | % | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_200Y101 | 주요지표(연간지표) | 국민계정 | 6,398 | 1 | C1:96 | C1:계정항목별 | 6 | %, 2020＝100, 달러, 만원, 십억원, 억달러 | A:73 | A 1953–2025 | 2026-09-20 |
| DT_200Y102 | 주요지표(분기지표) | 국민계정 | 10,632 | 1 | C1:53 | C1:계정항목별 | 2 | %, 십억원 | Q:266 | Q 196001–202602 | 2026-09-20 |
| DT_200Y103 | 경제활동별 GDP 및 GNI(계절조정, 명목, 분기) | 국민계정 | 12,198 | 1 | C1:51 | C1:계정항목별 | 1 | 십억원 | Q:266 | Q 196001–202602 | 2026-09-20 |
| DT_200Y104 | 경제활동별 GDP 및 GNI(계절조정, 실질, 분기) | 국민계정 | 12,730 | 1 | C1:53 | C1:계정항목별 | 1 | 십억원 | Q:266 | Q 196001–202602 | 2026-09-20 |
| DT_200Y105 | 경제활동별 GDP 및 GNI(원계열, 명목, 분기 및 연간) | 국민계정 | 13,102 | 1 | C1:55 | C1:계정항목별 | 1 | 십억원 | Q:266 | Q 196001–202602 | 2026-09-20 |
| DT_200Y106 | 경제활동별 GDP 및 GNI(원계열, 실질, 분기 및 연간) | 국민계정 | 13,634 | 1 | C1:57 | C1:계정항목별 | 1 | 십억원 | Q:266 | Q 196001–202602 | 2026-09-20 |
| DT_200Y107 | 국내총생산에 대한 지출(계절조정, 명목, 분기) | 국민계정 | 5,186 | 1 | C1:21 | C1:계정항목별 | 1 | 십억원 | Q:266 | Q 196001–202602 | 2026-09-20 |
| DT_200Y108 | 국내총생산에 대한 지출(계절조정, 실질, 분기) | 국민계정 | 5,186 | 1 | C1:21 | C1:계정항목별 | 1 | 십억원 | Q:266 | Q 196001–202602 | 2026-09-20 |
| DT_200Y109 | 국내총생산에 대한 지출(원계열, 명목, 분기 및 연간) | 국민계정 | 5,186 | 1 | C1:21 | C1:계정항목별 | 1 | 십억원 | Q:266 | Q 196001–202602 | 2026-09-20 |
| DT_200Y110 | 국내총생산에 대한 지출(원계열, 실질, 분기 및 연간) | 국민계정 | 5,186 | 1 | C1:21 | C1:계정항목별 | 1 | 십억원 | Q:266 | Q 196001–202602 | 2026-09-20 |
| DT_200Y111 | 경제활동별 국내총생산 디플레이터(분기 및 연간) | 국민계정 | 4,788 | 1 | C1:18 | C1:계정항목별 | 1 | 2020＝100 | Q:266 | Q 196001–202602 | 2026-09-20 |
| DT_200Y112 | 국내총생산에 대한 지출 디플레이터(분기 및 연간) | 국민계정 | 4,654 | 1 | C1:19 | C1:계정항목별 | 1 | 2020＝100 | Q:266 | Q 196001–202602 | 2026-09-20 |
| DT_200Y113 | 국내총생산과 지출(명목, 연간) | 국민계정 | 896 | 1 | C1:16 | C1:계정항목별 | 1 | 십억원 | A:56 | A 1970–2025 | 2026-09-20 |
| DT_200Y114 | 경제활동별 국내총부가가치와 요소소득(명목, 연간) | 국민계정 | 12,760 | 1 | C1:8, C2:29 | C1:금액구분코드별, C2:계정항목별 | 1 | 십억원 | A:55 | A 1970–2024 | 2026-09-20 |
| DT_200Y115 | 국민처분가능소득과 처분(명목, 연간) | 국민계정 | 896 | 1 | C1:16 | C1:계정항목별 | 1 | 십억원 | A:56 | A 1970–2025 | 2026-09-20 |
| DT_200Y116 | 국민소득과 부문별 국민처분가능소득(명목, 연간) | 국민계정 | 866 | 1 | C1:16 | C1:계정항목별 | 1 | 십억원 | A:56 | A 1970–2025 | 2026-09-20 |
| DT_200Y117 | 자본거래(명목, 연간) | 국민계정 | 530 | 1 | C1:11 | C1:계정항목별 | 1 | 십억원 | A:56 | A 1970–2025 | 2026-09-20 |
| DT_200Y118 | 경상거래(명목, 연간) | 국민계정 | 952 | 1 | C1:17 | C1:계정항목별 | 1 | 십억원 | A:56 | A 1970–2025 | 2026-09-20 |
| DT_200Y119 | 자본거래(명목, 연간) | 국민계정 | 336 | 1 | C1:7 | C1:계정항목별 | 1 | 십억원 | A:56 | A 1970–2025 | 2026-09-20 |
| DT_200Y120 | 제도부문별 생산계정(명목, 연간) | 국민계정 | 925 | 1 | C1:10, C2:9 | C1:금액구분코드별, C2:계정항목별 | 1 | 십억원 | A:25 | A 2000–2024 | 2026-09-20 |
| DT_200Y121 | 제도부문별 소득계정(명목, 연간) | 국민계정 | 16,732 | 1 | C1:12, C2:60 | C1:금액구분코드별, C2:계정항목별 | 1 | 십억원 | A:51 | A 1975–2025 | 2026-09-20 |
| DT_200Y122 | 제도부문별 자본계정(명목, 연간) | 국민계정 | 2,515 | 1 | C1:12, C2:10 | C1:금액구분코드별, C2:계정항목별 | 1 | 십억원 | A:51 | A 1975–2025 | 2026-09-20 |
| DT_200Y123 | 경제활동별 성장기여도(계절조정, 실질, 분기) | 국민계정 | 11,352 | 1 | C1:48 | C1:계정항목별 | 1 | %p | Q:265 | Q 196002–202602 | 2026-09-20 |
| DT_200Y124 | 경제활동별 성장기여도(원계열, 실질, 분기 및 연간) | 국민계정 | 12,096 | 1 | C1:52 | C1:계정항목별 | 1 | %p | Q:262 | Q 196101–202602 | 2026-09-20 |
| DT_200Y125 | 지출항목별 성장기여도(계절조정, 실질, 분기) | 국민계정 | 6,170 | 1 | C1:26 | C1:계정항목별 | 1 | %p | Q:265 | Q 196002–202602 | 2026-09-20 |
| DT_200Y126 | 지출항목별 성장기여도(원계열, 실질, 분기 및 연간) | 국민계정 | 6,092 | 1 | C1:26 | C1:계정항목별 | 1 | %p | Q:262 | Q 196101–202602 | 2026-09-20 |
| DT_200Y127 | 자본재형태별 총자본형성(계절조정, 명목, 분기) | 국민계정 | 3,164 | 1 | C1:14 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y128 | 자본재형태별 총자본형성(계절조정, 실질, 분기) | 국민계정 | 3,164 | 1 | C1:14 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y129 | 자본재형태별 총자본형성(원계열, 명목, 분기 및 연간) | 국민계정 | 3,164 | 1 | C1:14 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y130 | 자본재형태별 총자본형성(원계열, 실질, 분기 및 연간) | 국민계정 | 3,164 | 1 | C1:14 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y131 | 주체별 총자본형성(원계열, 명목, 분기 및 연간) | 국민계정 | 1,484 | 1 | C1:14 | C1:계정항목별 | 1 | 십억원 | Q:106 | Q 200001–202602 | 2026-09-20 |
| DT_200Y132 | 주체별 총자본형성(원계열, 실질, 분기 및 연간) | 국민계정 | 1,484 | 1 | C1:14 | C1:계정항목별 | 1 | 십억원 | Q:106 | Q 200001–202602 | 2026-09-20 |
| DT_200Y133 | 주체별 총고정자본형성(계절조정, 명목, 분기) | 국민계정 | 1,272 | 1 | C1:12 | C1:계정항목별 | 1 | 십억원 | Q:106 | Q 200001–202602 | 2026-09-20 |
| DT_200Y134 | 주체별 총고정자본형성(계절조정, 실질, 분기) | 국민계정 | 1,272 | 1 | C1:12 | C1:계정항목별 | 1 | 십억원 | Q:106 | Q 200001–202602 | 2026-09-20 |
| DT_200Y135 | 경제활동별 총고정자본형성(명목, 연간) | 국민계정 | 935 | 1 | C1:17 | C1:계정항목별 | 1 | 십억원 | A:55 | A 1970–2024 | 2026-09-20 |
| DT_200Y136 | 경제활동별 총고정자본형성(실질, 연간) | 국민계정 | 935 | 1 | C1:17 | C1:계정항목별 | 1 | 십억원 | A:55 | A 1970–2024 | 2026-09-20 |
| DT_200Y137 | 경제활동별 자본재형태별 총고정자본형성(명목, 연간) | 국민계정 | 2,380 | 1 | C1:7, C2:17 | C1:금액구분코드별, C2:계정항목별 | 1 | 십억원 | A:20 | A 2005–2024 | 2026-09-20 |
| DT_200Y138 | 경제활동별 설비투자(명목, 연간) | 국민계정 | 588 | 1 | C1:28 | C1:계정항목별 | 1 | 십억원 | A:21 | A 2005–2025 | 2026-09-20 |
| DT_200Y139 | 경제활동별 설비투자(실질, 연간) | 국민계정 | 588 | 1 | C1:28 | C1:계정항목별 | 1 | 십억원 | A:21 | A 2005–2025 | 2026-09-20 |
| DT_200Y140 | 가계의 목적별 최종소비지출(계절조정, 명목, 분기) | 국민계정 | 3,842 | 1 | C1:17 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y141 | 가계의 목적별 최종소비지출(계절조정, 실질, 분기) | 국민계정 | 3,842 | 1 | C1:17 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y142 | 가계의 목적별 최종소비지출(원계열, 명목, 분기 및 연간) | 국민계정 | 3,842 | 1 | C1:17 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y143 | 가계의 목적별 최종소비지출(원계열, 실질, 분기 및 연간) | 국민계정 | 3,842 | 1 | C1:17 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y144 | 가계의 형태별 최종소비지출(계절조정, 명목, 분기) | 국민계정 | 1,808 | 1 | C1:8 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y145 | 가계의 형태별 최종소비지출(계절조정, 실질, 분기) | 국민계정 | 1,808 | 1 | C1:8 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y146 | 가계의 형태별 최종소비지출(원계열, 명목, 분기 및 연간) | 국민계정 | 1,808 | 1 | C1:8 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y147 | 가계의 형태별 최종소비지출(원계열, 실질, 분기 및 연간) | 국민계정 | 1,808 | 1 | C1:8 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y148 | 가계에 봉사하는 비영리단체의 목적별 최종소비지출(원계열, 명목, 분기 및 연간) | 국민계정 | 1,356 | 1 | C1:6 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y149 | 가계에 봉사하는 비영리단체의 목적별 최종소비지출(원계열, 실질, 분기 및 연간) | 국민계정 | 1,356 | 1 | C1:6 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y151 | 일반정부의 부문별 기능별 최종소비지출(명목, 연간) | 국민계정 | 26,620 | 1 | C1:44, C2:11 | C1:금액구분코드별, C2:계정항목별 | 1 | 십억원 | A:55 | A 1970–2024 | 2026-09-20 |
| DT_200Y152 | 일반정부의 부문별 기능별 총지출 (명목, 연간) | 국민계정 | 31,460 | 1 | C1:52, C2:11 | C1:금액구분코드별, C2:계정항목별 | 1 | 십억원 | A:55 | A 1970–2024 | 2026-09-20 |
| DT_200Y153 | 일반정부의 부문별 총수입, 총지출, 저축투자차액(명목, 연간) | 국민계정 | 11,200 | 1 | C1:4, C2:50 | C1:금액구분코드별, C2:계정항목별 | 1 | 십억원 | A:56 | A 1970–2025 | 2026-09-20 |
| DT_200Y154 | 공공부문의 부문별 총수입, 총지출, 저축투자차액(명목, 연간) | 국민계정 | 5,400 | 1 | C1:6, C2:50 | C1:금액구분코드별, C2:계정항목별 | 1 | 십억원 | A:18 | A 2007–2025 | 2026-09-20 |
| DT_200Y155 | 총저축과 총투자(계절조정, 명목, 분기) | 국민계정 | 678 | 1 | C1:3 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y156 | 총저축과 총투자(원계열, 명목, 분기 및 연간) | 국민계정 | 678 | 1 | C1:3 | C1:계정항목별 | 1 | 십억원 | Q:226 | Q 197001–202602 | 2026-09-20 |
| DT_200Y157 | 정보통신산업(계절조정, 실질, 분기) | 국민계정 | 1,220 | 1 | C1:10 | C1:계정항목별 | 1 | 십억원 | Q:126 | Q 199501–202602 | 2026-09-20 |
| DT_200Y158 | 정보통신산업(원계열, 실질, 분기 및 연간) | 국민계정 | 1,220 | 1 | C1:10 | C1:계정항목별 | 1 | 십억원 | Q:126 | Q 199501–202602 | 2026-09-20 |
| DT_200Y160 | 부가가치 항목별 명목 GDP(계절조정, 명목, 분기) | 국민계정 | 264 | 1 | C1:4 | C1:계정항목별 | 1 | 십억원 | Q:66 | Q 201001–202602 | 2026-09-20 |
| DT_200Y161 | 부가가치 항목별 명목 GDP(원계열, 명목, 분기 및 연간) | 국민계정 | 264 | 1 | C1:4 | C1:계정항목별 | 1 | 십억원 | Q:66 | Q 201001–202602 | 2026-09-20 |
| DT_200Y162 | 제도부문별 소득계정(명목, 분기) | 국민계정 | 5,184 | 1 | C1:12, C2:12 | C1:금액구분코드별, C2:계정항목별 | 1 | 십억원 | Q:64 | Q 201001–202504 | 2026-09-20 |
| DT_281Y001 | 금융거래표(2009~) | 자금순환표 | 69 | 1 | C1:1, C2:1, C3:1 | C1:제도부문코드별, C2:계정코드별, C3:운용원천구분코드별 | 1 | 십억원 | Q:69 | Q 200901–202601 | 2026-09-20 |
| DT_281Y002 | 금융자산부채잔액표(2008~) | 자금순환표 | 70 | 1 | C1:1, C2:1, C3:1 | C1:제도부문코드별, C2:계정코드별, C3:자산/부채별 | 1 | 십억원 | Q:70 | Q 200804–202601 | 2026-09-20 |
| DT_281Y004 | 금융거래표(1975~2005) | 자금순환표 | 124 | 1 | C1:1, C2:1, C3:1 | C1:계정항목별, C2:운용원천구분코드별, C3:제도부문코드별 | 1 | 십억원 | Q:124 | Q 197501–200504 | — |
| DT_281Y005 | 금융자산부채잔액표(1975~2005) | 자금순환표 | 124 | 1 | C1:1, C2:1, C3:1 | C1:계정항목별, C2:자산/부채별, C3:제도부문코드별 | 1 | 십억원 | Q:124 | Q 197501–200504 | — |
| DT_281Y006 | 금융거래표(2003~2013) | 자금순환표 | 44 | 1 | C1:1, C2:1, C3:1 | C1:계정항목별, C2:운용원천구분코드별, C3:제도부문코드별 | 1 | 십억원 | Q:44 | Q 200301–201304 | — |
| DT_281Y007 | 금융자산부채잔액표(2002~) | 자금순환표 | 45 | 1 | C1:1, C2:1, C3:1 | C1:계정항목별, C2:자산/부채별, C3:제도부문코드별 | 1 | 십억원 | Q:45 | Q 200204–201304 | — |
| DT_282Y001 | 발행잔액(명목) | 자금순환표 | 3,960 | 1 | C1:12, C2:10 | C1:기관코드별, C2:계정항목별 | 1 | 십억원 | Q:33 | Q 201801–202601 | 2026-09-20 |
| DT_282Y002 | 발행잔액(시가) | 자금순환표 | 3,960 | 1 | C1:12, C2:10 | C1:기관코드별, C2:계정항목별 | 1 | 십억원 | Q:33 | Q 201801–202601 | 2026-09-20 |
| DT_282Y003 | 순발행액(시가) | 자금순환표 | 3,960 | 1 | C1:12, C2:10 | C1:기관코드별, C2:계정항목별 | 1 | 십억원 | Q:33 | Q 201801–202601 | 2026-09-20 |
| DT_282Y004 | 보유잔액(시가) | 자금순환표 | 9,459 | 1 | C1:14, C2:21 | C1:기관코드별, C2:계정항목별 | 1 | 십억원 | Q:33 | Q 201801–202601 | 2026-09-20 |
| DT_282Y005 | 순취득액(시가) | 자금순환표 | 9,504 | 1 | C1:14, C2:21 | C1:기관코드별, C2:계정항목별 | 1 | 십억원 | Q:33 | Q 201801–202601 | 2026-09-20 |
| DT_282Y006 | 채권발행-보유관계표(잔액, 시가) | 자금순환표 | 1,584 | 1 | C1:7, C2:7 | C1:발행(행)별, C2:보유(열)별 | 1 | 십억원 | Q:33 | Q 201801–202601 | 2026-09-20 |
| DT_284Y001 | 잔액표 | 자금순환표 | CSV 없음 | — | — | — | — | — | — | — | — |
| DT_284Y002 | 거래표 | 자금순환표 | CSV 없음 | — | — | — | — | — | — | — | — |
| DT_301Y013 | 국제수지 | 국제수지통계 | 559 | 1 | C1:1 | C1:계정코드별 | 1 | 백만달러 | M:559 | M 198001–202607 | 2026-09-20 |
| DT_301Y014 | 서비스무역세분류통계 | 국제수지통계 | 36,062 | 1 | C1:146 | C1:계정코드별 | 1 | 백만달러 | M:247 | M 200601–202607 | 2026-09-20 |
| DT_301Y015 | 지역별 경상수지 | 국제수지통계 | 14,112 | 1 | C1:56, C2:9 | C1:계정코드별, C2:지역코드별 | 1 | 백만달러 | A:28 | A 1998–2025 | 2026-09-20 |
| DT_301Y016 | 지역별 자본수지/금융계정 | 국제수지통계 | 2,880 | 1 | C1:16, C2:9 | C1:계정코드별, C2:지역코드별 | 1 | 백만달러 | A:20 | A 2006–2025 | 2026-09-20 |
| DT_301Y017 | 경상수지(계절조정) | 국제수지통계 | 10,621 | 1 | C1:19 | C1:계정코드별 | 1 | 백만달러 | M:559 | M 198001–202607 | 2026-09-20 |
| DT_303Y001 | 지역별 결제통화(수출) | 국제수지통계 | 58,332 | 1 | C1:82, C2:2 | C1:계정코드별, C2:금액/비중 구분별 | 2 | %, 백만달러 | M:408 | M 199201–202512 | 2026-09-20 |
| DT_303Y002 | 국가별 결제통화(수출) | 국제수지통계 | 42,684 | 1 | C1:61, C2:2 | C1:계정코드별, C2:금액/비중 구분별 | 2 | %, 백만달러 | M:408 | M 199201–202512 | 2026-09-20 |
| DT_303Y003 | 지역별,국가별 원화결제(수출) | 국제수지통계 | 19,442 | 1 | C1:28, C2:2 | C1:계정코드별, C2:금액/비중 구분별 | 2 | %, 백만달러 | M:408 | M 199201–202512 | 2026-09-20 |
| DT_303Y004 | 지역별 결제통화(수입) | 국제수지통계 | 58,572 | 1 | C1:82, C2:2 | C1:계정코드별, C2:금액/비중 구분별 | 2 | %, 백만달러 | M:408 | M 199201–202512 | 2026-09-20 |
| DT_303Y005 | 국가별 결제통화(수입) | 국제수지통계 | 42,208 | 1 | C1:61, C2:2 | C1:계정코드별, C2:금액/비중 구분별 | 2 | %, 백만달러 | M:408 | M 199201–202512 | 2026-09-20 |
| DT_303Y006 | 지역별,국가별 원화결제(수입) | 국제수지통계 | 18,024 | 1 | C1:25, C2:2 | C1:계정코드별, C2:금액/비중 구분별 | 2 | %, 백만달러 | M:408 | M 199201–202512 | 2026-09-20 |
| DT_304Y101 | 지식서비스 무역통계(유형별) | 국제수지통계 | 4,950 | 1 | C1:3, C2:25 | C1:구분코드별, C2:계정항목별 | 1 | 백만달러 | Q:66 | Q 201001–202602 | 2026-09-20 |
| DT_304Y102 | 지식서비스 무역통계(산업별) | 국제수지통계 | 7,722 | 1 | C1:3, C2:39 | C1:구분코드별, C2:계정항목별 | 1 | 백만달러 | Q:66 | Q 201001–202602 | 2026-09-20 |
| DT_304Y103 | 지식서비스 무역통계(기관형태별) | 국제수지통계 | 1,584 | 1 | C1:3, C2:8 | C1:구분코드별, C2:계정항목별 | 1 | 백만달러 | Q:66 | Q 201001–202602 | 2026-09-20 |
| DT_304Y104 | 지식서비스 무역통계(지역별) | 국제수지통계 | 4,950 | 1 | C1:3, C2:25 | C1:구분코드별, C2:계정항목별 | 1 | 백만달러 | Q:66 | Q 201001–202602 | 2026-09-20 |
| DT_304Y105 | ICT 산업 특수분류(ICT산업별) | 국제수지통계 | 14,058 | 1 | C1:3, C2:71 | C1:구분코드별, C2:계정항목별 | 1 | 백만달러 | Q:66 | Q 201001–202602 | 2026-09-20 |
| DT_304Y106 | 콘텐츠 산업 특수분류(콘텐츠산업별) | 국제수지통계 | 10,494 | 1 | C1:3, C2:53 | C1:구분코드별, C2:계정항목별 | 1 | 백만달러 | Q:66 | Q 201001–202602 | 2026-09-20 |
| DT_304Y107 | 지식재산권 무역수지(유형별) | 국제수지통계 | 3,168 | 1 | C1:3, C2:16 | C1:구분코드별, C2:계정항목별 | 1 | 백만달러 | Q:66 | Q 201001–202602 | 2026-09-20 |
| DT_311Y001 | 국제투자대조표(IIP) | 국제투자대조표(IIP) | 25,157 | 1 | C1:211 | C1:계정코드별 | 1 | 백만달러 | Q:127 | Q 199404–202602 | 2026-09-20 |
| DT_311Y002 | 지역별 국제투자대조표 | 국제투자대조표(IIP) | 3,024 | 1 | C1:14, C2:9 | C1:계정코드별, C2:지역코드별 | 1 | 백만달러 | A:24 | A 2002–2025 | 2026-09-20 |
| DT_311Y003 | 통화별 국제투자대조표 | 국제투자대조표(IIP) | 3,024 | 1 | C1:14, C2:9 | C1:계정코드별, C2:통화별 | 1 | 백만달러 | A:24 | A 2002–2025 | 2026-09-20 |
| DT_311Y004 | 대외채무 | 대외채무및대외채권 | 12,446 | 1 | C1:98 | C1:계정코드별 | 1 | 백만달러 | Q:127 | Q 199404–202602 | 2026-09-20 |
| DT_311Y005 | 대외채권 | 대외채무및대외채권 | 12,319 | 1 | C1:97 | C1:계정코드별 | 1 | 백만달러 | Q:127 | Q 199404–202602 | 2026-09-20 |
| DT_311Y006 | 순대외채권 | 대외채무및대외채권 | 2,667 | 1 | C1:21 | C1:계정코드별 | 1 | 백만달러 | Q:127 | Q 199404–202602 | 2026-09-20 |
| DT_401Y015 | 수입물가지수(기본분류) | 수출입물가조사(2020=100) | 668 | 1 | C1:1, C2:1 | C1:계정코드별, C2:통화계약구분코드별 | 1 | 2020＝100 | M:668 | M 197101–202608 | 2026-09-20 |
| DT_401Y016 | 수입물가지수(특수분류) | 수출입물가조사(2020=100) | 668 | 1 | C1:1, C2:1 | C1:계정코드별, C2:통화계약구분코드별 | 1 | 2020＝100 | M:668 | M 197101–202608 | 2026-09-20 |
| DT_401Y017 | 수입물가지수(품목별) | 수출입물가조사(2020=100) | 668 | 1 | C1:1, C2:1 | C1:계정코드별, C2:통화계약구분코드별 | 1 | 2020＝100 | M:668 | M 197101–202608 | 2026-09-20 |
| DT_401Y018 | 수입물가지수(용도별) | 수출입물가조사(2020=100) | 668 | 1 | C1:1, C2:1 | C1:계정코드별, C2:통화계약구분코드별 | 1 | 2020＝100 | M:668 | M 197101–202608 | 2026-09-20 |
| DT_402Y014 | 수출물가지수(기본분류) | 수출입물가조사(2020=100) | 668 | 1 | C1:1, C2:1 | C1:계정코드별, C2:통화계약구분코드별 | 1 | 2020＝100 | M:668 | M 197101–202608 | 2026-09-20 |
| DT_402Y015 | 수출물가지수(특수분류) | 수출입물가조사(2020=100) | 668 | 1 | C1:1, C2:1 | C1:계정코드별, C2:통화계약구분코드별 | 1 | 2020＝100 | M:668 | M 197101–202608 | 2026-09-20 |
| DT_402Y016 | 수출물가지수(품목별) | 수출입물가조사(2020=100) | 668 | 1 | C1:1, C2:1 | C1:계정코드별, C2:통화계약구분코드별 | 1 | 2020＝100 | M:668 | M 197101–202608 | 2026-09-20 |
| DT_403Y001 | 수출금액지수 | 수출입물가조사(2020=100) | 464 | 1 | C1:1 | C1:계정코드별 | 1 | 2020＝100 | M:464 | M 198801–202608 | 2026-09-20 |
| DT_403Y002 | 수출물량지수 | 수출입물가조사(2020=100) | 464 | 1 | C1:1 | C1:계정코드별 | 1 | 2020＝100 | M:464 | M 198801–202608 | 2026-09-20 |
| DT_403Y003 | 수입금액지수 | 수출입물가조사(2020=100) | 464 | 1 | C1:1 | C1:계정코드별 | 1 | 2020＝100 | M:464 | M 198801–202608 | 2026-09-20 |
| DT_403Y004 | 수입물량지수 | 수출입물가조사(2020=100) | 464 | 1 | C1:1 | C1:계정코드별 | 1 | 2020＝100 | M:464 | M 198801–202608 | 2026-09-20 |
| DT_403Y005 | 교역조건지수 | 수출입물가조사(2020=100) | 928 | 1 | C1:2 | C1:교역조건구분별 | 1 | 2020＝100 | M:464 | M 198801–202608 | 2026-09-20 |
| DT_404Y014 | 생산자물가지수(기본분류) | 생산자물가조사(2020=100) | 740 | 1 | C1:1 | C1:계정코드별 | 1 | 2020＝100 | M:740 | M 196501–202608 | 2026-09-20 |
| DT_404Y015 | 생산자물가지수(특수분류) | 생산자물가조사(2020=100) | 6,580 | 1 | C1:11 | C1:계정코드별 | 1 | 2020＝100 | M:740 | M 196501–202608 | 2026-09-20 |
| DT_404Y016 | 생산자물가지수(품목별) | 생산자물가조사(2020=100) | 740 | 1 | C1:1 | C1:계정코드별 | 1 | 2020＝100 | M:740 | M 196501–202608 | 2026-09-20 |
| DT_404Y017 | 생산자물가지수-총지수(1910~1964) | 생산자물가조사(2020=100) | 654 | 1 | C1:1 | C1: | 1 | 2020＝100 | M:654 | M 191007–196412 | 2024-05-14 |
| DT_405Y006 | 국내공급물가지수 | 생산자물가조사(2020=100) | 31,002 | 1 | C1:59 | C1:계정코드별 | 2 | 2020＝100, 2022.12＝100 | M:560 | M 198001–202608 | 2026-09-20 |
| DT_405Y007 | 총산출물가지수 | 생산자물가조사(2020=100) | 12,800 | 1 | C1:64 | C1:계정코드별 | 1 | 2020＝100 | M:200 | M 201001–202608 | 2026-09-20 |
| DT_501Y001 | 재무상태표(제11차 한국표준산업분류, 2009~) | 기업경영분석 | 16 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 백만원 | A:16 | A 2009–2024 | 2026-09-20 |
| DT_501Y002 | 손익계산서(제11차 한국표준산업분류, 2009~) | 기업경영분석 | 16 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 백만원 | A:16 | A 2009–2024 | 2026-09-20 |
| DT_501Y003 | 제조원가명세서(제11차 한국표준산업분류, 2009~) | 기업경영분석 | 16 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 백만원 | A:16 | A 2009–2024 | 2026-09-20 |
| DT_501Y004 | 이익잉여금 처분계산서(제11차 한국표준산업분류, 2009~) | 기업경영분석 | 16 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:16 | A 2009–2024 | 2026-09-20 |
| DT_501Y005 | 성장성 지표(제11차 한국표준산업분류, 2009~) | 기업경영분석 | 15 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:15 | A 2010–2024 | 2026-09-20 |
| DT_501Y006 | 손익 지표(제11차 한국표준산업분류, 2009~) | 기업경영분석 | 16 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:16 | A 2009–2024 | 2026-09-20 |
| DT_501Y007 | 자산/자본 지표(제11차 한국표준산업분류, 2009~) | 기업경영분석 | 16 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:16 | A 2009–2024 | 2026-09-20 |
| DT_501Y008 | 자산/자본 회전율(제11차 한국표준산업분류, 2009~) | 기업경영분석 | 16 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 회 | A:16 | A 2009–2024 | 2026-09-20 |
| DT_501Y009 | 생산성 지표(제11차 한국표준산업분류, 2009~) | 기업경영분석 | 16 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:16 | A 2009–2024 | 2026-09-20 |
| DT_501Y010 | 부가가치 구성(제11차 한국표준산업분류, 2009~) | 기업경영분석 | 16 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:16 | A 2009–2024 | 2026-09-20 |
| DT_501Y013 | 현금흐름표(2012~) | 기업경영분석 | 2 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기준년도구분별, C3:계정항목별 | 1 | 백만원 | A:2 | A 2023–2024 | 2026-09-20 |
| DT_501Y014 | 현금흐름관련 기타 주요 재무항목(2012~) | 기업경영분석 | 2 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기준년도구분별, C3:계정항목별 | 1 | 백만원 | A:2 | A 2023–2024 | 2026-09-20 |
| DT_501Y015 | 현금흐름비율(2012~) | 기업경영분석 | 2 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기준년도구분별, C3:계정항목별 | 1 | % | A:2 | A 2023–2024 | 2026-09-20 |
| DT_501Y017 | 기업경영분석지표(2007~2010) | 기업경영분석 | 120 | 1 | C1:2, C2:15 | C1:업종코드별, C2:계정항목별 | — | 열 없음 | A:4 | A 2007–2010 | — |
| DT_501Y018 | 기업경영분석지표(~2007) | 기업경영분석 | 820 | 1 | C1:2, C2:17 | C1:업종코드별, C2:계정항목별 | — | 열 없음 | A:48 | A 1960–2007 | — |
| DT_501Y019 | 성장성 지표(중소기업기본법, 2015~2016) | 기업경영분석 | 2,652 | 1 | C1:139, C2:3, C3:6 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:2 | A 2015–2016 | — |
| DT_501Y020 | 성장성 지표(조세특례제한법, 2009~2014) | 기업경영분석 | 7,948 | 1 | C1:139, C2:3, C3:6 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:6 | A 2009–2014 | — |
| DT_501Y021 | 성장성 지표(2007~2010, 표본조사) | 기업경영분석 | 4,830 | 1 | C1:129, C2:3, C3:6 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:4 | A 2007–2010 | — |
| DT_501Y022 | 성장성 지표(1998~2007, 표본조사) | 기업경영분석 | 6 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:6 | A 2002–2007 | — |
| DT_501Y023 | 성장성 지표(1990~1997, 표본조사) | 기업경영분석 | 13,888 | 1 | C1:151, C2:3, C3:8 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:8 | A 1990–1997 | — |
| DT_501Y024 | 손익 지표(중소기업기본법, 2015~2016) | 기업경영분석 | 15,018 | 1 | C1:139, C2:3, C3:34 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:2 | A 2015–2016 | — |
| DT_501Y025 | 손익 지표(조세특례제한법, 2009~2014) | 기업경영분석 | 6 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:6 | A 2009–2014 | — |
| DT_501Y026 | 손익 지표(2007~2010, 표본조사) | 기업경영분석 | 4 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:4 | A 2007–2010 | — |
| DT_501Y027 | 손익 지표(1998~2007, 표본조사) | 기업경영분석 | 6 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:6 | A 2002–2007 | — |
| DT_501Y028 | 손익 지표(1990~1997, 표본조사) | 기업경영분석 | 8 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:8 | A 1990–1997 | — |
| DT_501Y029 | 자산/자본 지표(중소기업기본법, 2015~2016) | 기업경영분석 | 5,746 | 1 | C1:139, C2:3, C3:13 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:2 | A 2015–2016 | — |
| DT_501Y030 | 자산/자본 지표(조세특례제한법, 2009~2014) | 기업경영분석 | 6 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:6 | A 2009–2014 | — |
| DT_501Y031 | 자산/자본 지표(2007~2010, 표본조사) | 기업경영분석 | 10,465 | 1 | C1:129, C2:3, C3:13 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:4 | A 2007–2010 | — |
| DT_501Y032 | 자산/자본 지표(1998~2007, 표본조사) | 기업경영분석 | 6 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:6 | A 2002–2007 | — |
| DT_501Y033 | 자산/자본 지표(1990~1997, 표본조사) | 기업경영분석 | 8 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:8 | A 1990–1997 | — |
| DT_501Y034 | 자산/자본 회전율(중소기업기본법, 2015~2016) | 기업경영분석 | 4,420 | 1 | C1:139, C2:3, C3:10 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 회 | A:2 | A 2015–2016 | — |
| DT_501Y035 | 자산/자본 회전율(조세특례제한법, 2009~2014) | 기업경영분석 | 13,260 | 1 | C1:139, C2:3, C3:10 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 회 | A:6 | A 2009–2014 | — |
| DT_501Y036 | 자산/자본 회전율(2007~2010, 표본조사) | 기업경영분석 | 8,050 | 1 | C1:129, C2:3, C3:10 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 회 | A:4 | A 2007–2010 | — |
| DT_501Y037 | 자산/자본 회전율(1998~2007, 표본조사) | 기업경영분석 | 6 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:6 | A 2002–2007 | — |
| DT_501Y038 | 자산/자본 회전율(1990~1997, 표본조사) | 기업경영분석 | 8 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:8 | A 1990–1997 | — |
| DT_501Y039 | 생산성 지표(중소기업기본법, 2015~2016) | 기업경영분석 | 2,210 | 1 | C1:139, C2:3, C3:5 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:2 | A 2015–2016 | — |
| DT_501Y040 | 생산성 지표(조세특례제한법, 2009~2014) | 기업경영분석 | 6,630 | 1 | C1:139, C2:3, C3:5 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:6 | A 2009–2014 | — |
| DT_501Y041 | 생산성 지표(2007~2010, 표본조사) | 기업경영분석 | 6,440 | 1 | C1:129, C2:3, C3:8 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 2 | %, 백만원 | A:4 | A 2007–2010 | — |
| DT_501Y042 | 생산성 지표(2002~2007, 표본조사) | 기업경영분석 | 9,202 | 1 | C1:150, C2:3, C3:8 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:6 | A 2002–2007 | — |
| DT_501Y043 | 생산성 지표(1998~2006, 표본조사) | 기업경영분석 | 5 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:5 | A 2002–2006 | — |
| DT_501Y044 | 생산성 지표(1990~1997, 표본조사) | 기업경영분석 | 8 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:8 | A 1990–1997 | — |
| DT_501Y049 | 재무상태표(중소기업기본법, 2015~2016) | 기업경영분석 | 2 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 백만원 | A:2 | A 2015–2016 | — |
| DT_501Y050 | 재무상태표(조세특례제한법, 2009~2014) | 기업경영분석 | 6 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 백만원 | A:6 | A 2009–2014 | — |
| DT_501Y051 | 재무상태표(2008~2010, 표본조사) | 기업경영분석 | 3 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 백만원 | A:3 | A 2008–2010 | — |
| DT_501Y052 | 재무상태표(2000~2007, 표본조사) | 기업경영분석 | 4 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:4 | A 2004–2007 | — |
| DT_501Y053 | 재무상태표(1999, 표본조사) | 기업경영분석 | 7,518 | 1 | C1:115, C2:3, C3:42 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 백만원 | A:1 | A 1999–1999 | — |
| DT_501Y054 | 재무상태표(1997~1998, 표본조사) | 기업경영분석 | 2 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 백만원 | A:2 | A 1997–1998 | — |
| DT_501Y055 | 재무상태표(1990~1996, 표본조사) | 기업경영분석 | 7 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:7 | A 1990–1996 | — |
| DT_501Y056 | 손익계산서(중소기업기본법, 2015~2016) | 기업경영분석 | 15,912 | 1 | C1:139, C2:3, C3:36 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 백만원 | A:2 | A 2015–2016 | — |
| DT_501Y057 | 손익계산서(조세특례제한법, 2009~2014) | 기업경영분석 | 6 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 백만원 | A:6 | A 2009–2014 | — |
| DT_501Y058 | 손익계산서(2008~2010, 표본조사) | 기업경영분석 | 3 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 백만원 | A:3 | A 2008–2010 | — |
| DT_501Y059 | 손익계산서(2000~2007, 표본조사) | 기업경영분석 | 4 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:4 | A 2004–2007 | — |
| DT_501Y060 | 손익계산서(1999, 표본조사) | 기업경영분석 | 7,518 | 1 | C1:115, C2:3, C3:42 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 백만원 | A:1 | A 1999–1999 | — |
| DT_501Y061 | 손익계산서(1997~1998, 표본조사) | 기업경영분석 | 2 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:2 | A 1997–1998 | — |
| DT_501Y062 | 손익계산서(1990~1996, 표본조사) | 기업경영분석 | 7 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:7 | A 1990–1996 | — |
| DT_501Y063 | 제조원가명세서(중소기업기본법, 2015~2016) | 기업경영분석 | 8,840 | 1 | C1:139, C2:3, C3:20 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 백만원 | A:2 | A 2015–2016 | — |
| DT_501Y064 | 제조원가명세서(조세특례제한법, 2009~2014) | 기업경영분석 | 6 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 백만원 | A:6 | A 2009–2014 | — |
| DT_501Y065 | 제조원가명세서(2007~2010, 표본조사) | 기업경영분석 | 12,080 | 1 | C1:129, C2:3, C3:20 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | 백만원 | A:3 | A 2008–2010 | — |
| DT_501Y066 | 제조원가명세서(2000~2007, 표본조사) | 기업경영분석 | 4 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:4 | A 2004–2007 | — |
| DT_501Y067 | 제조원가명세서(1990~1999, 표본조사) | 기업경영분석 | 10 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:10 | A 1990–1999 | — |
| DT_501Y068 | 이익잉여금 처분계산서(중소기업기본법, 2015~2016) | 기업경영분석 | 6,188 | 1 | C1:139, C2:3, C3:14 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:2 | A 2015–2016 | — |
| DT_501Y069 | 이익잉여금 처분계산서(조세특례제한법, 2009~2014) | 기업경영분석 | 6 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:6 | A 2009–2014 | — |
| DT_501Y070 | 이익잉여금 처분계산서(2007~2010, 표본조사) | 기업경영분석 | 12,030 | 1 | C1:129, C2:3, C3:15 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:4 | A 2007–2010 | — |
| DT_501Y071 | 이익잉여금 처분계산서(2000~2007, 표본조사) | 기업경영분석 | 6 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:6 | A 2002–2007 | — |
| DT_501Y072 | 이익잉여금 처분계산서(1990~1999, 표본조사) | 기업경영분석 | 10 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:10 | A 1990–1999 | — |
| DT_501Y073 | 부가가치 구성(중소기업기본법, 2015~2016) | 기업경영분석 | 2,210 | 1 | C1:139, C2:3, C3:5 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:2 | A 2015–2016 | — |
| DT_501Y074 | 부가가치 구성(조세특례제한법, 2009~2014) | 기업경영분석 | 6,630 | 1 | C1:139, C2:3, C3:5 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:6 | A 2009–2014 | — |
| DT_501Y075 | 부가가치 구성(2007~2010, 표본조사) | 기업경영분석 | 4,025 | 1 | C1:129, C2:3, C3:5 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | 1 | % | A:4 | A 2007–2010 | — |
| DT_501Y076 | 부가가치 구성(2002~2007, 표본조사) | 기업경영분석 | 5,710 | 1 | C1:147, C2:3, C3:5 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:6 | A 2002–2007 | — |
| DT_501Y077 | 부가가치 구성(1998~2006, 표본조사) | 기업경영분석 | 10,182 | 1 | C1:150, C2:3, C3:6 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:9 | A 1998–2006 | — |
| DT_501Y078 | 부가가치 구성(1990~1997, 표본조사) | 기업경영분석 | 10,416 | 1 | C1:151, C2:3, C3:6 | C1:업종코드별, C2:기업규모별, C3:계정항목별 | — | 열 없음 | A:8 | A 1990–1997 | — |
| DT_501Y079 | 현금흐름표(2008~2011) | 기업경영분석 | 12,288 | 1 | C1:32, C2:4, C3:48 | C1:업종코드별, C2:기준년도구분별, C3:계정항목별 | — | 열 없음 | A:5 | A 2007–2011 | — |
| DT_501Y080 | 현금흐름표(1995~2007) | 기업경영분석 | 2 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기준년도구분별, C3:계정항목별 | — | 열 없음 | A:2 | A 2006–2007 | — |
| DT_501Y081 | 현금흐름관련 기타 주요 재무항목(2008~2011) | 기업경영분석 | 2,048 | 1 | C1:32, C2:4, C3:8 | C1:업종코드별, C2:기준년도구분별, C3:계정항목별 | — | 열 없음 | A:5 | A 2007–2011 | — |
| DT_501Y082 | 현금흐름관련 기타 주요 재무항목(1995~2007) | 기업경영분석 | 2 | 1 | C1:1, C2:1, C3:1 | C1:업종코드별, C2:기준년도구분별, C3:계정항목별 | — | 열 없음 | A:2 | A 2006–2007 | — |
| DT_501Y083 | 현금흐름비율(2008~2011) | 기업경영분석 | 2,048 | 1 | C1:32, C2:4, C3:8 | C1:업종코드별, C2:기준년도구분별, C3:계정항목별 | — | 열 없음 | A:5 | A 2007–2011 | — |
| DT_501Y084 | 현금흐름비율(1995~2007) | 기업경영분석 | 3,360 | 1 | C1:31, C2:7, C3:8 | C1:업종코드별, C2:기준년도구분별, C3:계정항목별 | — | 열 없음 | A:10 | A 1998–2007 | — |
| DT_501Y086 | 분위수(제11차 한국표준산업분류, 2013~) | 기업경영분석 | 12 | 1 | C1:1, C2:1, C3:1, C4:1 | C1:업종코드별, C2:기업규모별, C3:계정항목별, C4:분위 구분코드별 | 1 | % | A:12 | A 2013–2024 | 2026-09-20 |
| DT_511Y001 | 소비자동향조사(전국, 분기, 1995.3/4~2008.2/4) | 소비자동향조사 | 24,214 | 1 | C1:20, C2:27 | C1:CSI코드별, C2:CSI분류코드별 | — | 열 없음 | Q:52 | Q 199503–200802 | — |
| DT_511Y002 | 소비자동향조사(전국, 월, 2008.9~) | 소비자동향조사 | 117,698 | 1 | C1:30, C2:22 | C1:CSI코드별, C2:CSI분류코드별 | — | 열 없음 | M:218 | M 200807–202608 | 2026-09-20 |
| DT_511Y003 | 기대인플레이션율(전국, 월) | 소비자동향조사 | 4,810 | 1 | C1:55 | C1:CSI코드별 | 1 | % | M:295 | M 200202–202608 | 2026-09-20 |
| DT_511Y004 | 소비자동향조사(지역, 월) | 소비자동향조사 | 68,282 | 1 | C1:29, C2:13 | C1:CSI코드별, C2:담당부서별 | — | 열 없음 | M:218 | M 200807–202608 | 2026-09-20 |
| DT_512Y001 | 업종별 기업경기실사지수, ~2009.8(실적) | 기업경기조사 | 79 | 1 | C1:1, C2:1 | C1:업종코드별, C2:BSI코드별 | — | 열 없음 | M:79 | M 200301–200907 | — |
| DT_512Y002 | 업종별 기업경기실사지수, ~2009.8(전망) | 기업경기조사 | 79 | 1 | C1:1, C2:1 | C1:업종코드별, C2:BSI코드별 | — | 열 없음 | M:79 | M 200302–200908 | — |
| DT_512Y007 | 업종별 기업경기실사지수, 2009.8~(실적) | 기업경기조사 | 205 | 1 | C1:1, C2:1 | C1:BSI코드별, C2:업종코드별 | — | 열 없음 | M:205 | M 200908–202608 | 2026-09-20 |
| DT_512Y008 | 업종별 기업경기실사지수, 2009.9~(전망) | 기업경기조사 | 205 | 1 | C1:1, C2:1 | C1:BSI코드별, C2:업종코드별 | — | 열 없음 | M:205 | M 200909–202609 | 2026-09-20 |
| DT_512Y011 | 지역별 기업경기실사지수(실적(~2012.12)) | 기업경기조사 | 24,238 | 1 | C1:22, C2:12 | C1:BSI코드별, C2:담당부서별 | — | 열 없음 | M:120 | M 200301–201212 | — |
| DT_512Y012 | 지역별 기업경기실사지수(전망(~2013.1)) | 기업경기조사 | 22,546 | 1 | C1:22, C2:12 | C1:BSI코드별, C2:담당부서별 | — | 열 없음 | M:121 | M 200301–201301 | — |
| DT_512Y013 | 기업경기조사(실적) | 기업경기조사 | 284 | 1 | C1:1, C2:1 | C1:업종코드별, C2:BSI코드별 | — | 열 없음 | M:284 | M 200301–202608 | 2026-09-20 |
| DT_512Y014 | 기업경기조사(전망) | 기업경기조사 | 284 | 1 | C1:1, C2:1 | C1:업종코드별, C2:BSI코드별 | — | 열 없음 | M:284 | M 200302–202609 | 2026-09-20 |
| DT_512Y015 | 기업경기조사(매출액가중 실적) | 기업경기조사 | 5,098 | 1 | C1:3, C2:15 | C1:업종코드별, C2:BSI코드별 | — | 열 없음 | M:205 | M 200908–202608 | 2026-09-20 |
| DT_512Y016 | 기업경기조사(매출액가중 전망) | 기업경기조사 | 5,098 | 1 | C1:3, C2:15 | C1:업종코드별, C2:BSI코드별 | — | 열 없음 | M:205 | M 200909–202609 | 2026-09-20 |
| DT_512Y017 | 기업경기실사지수((구)실적(~2012.6)) | 기업경기조사 | 13,338 | 1 | C1:9, C2:16 | C1:업종코드별, C2:BSI코드별 | — | 열 없음 | M:114 | M 200301–201206 | — |
| DT_512Y018 | 기업경기실사지수((구)전망(~2012.7)) | 기업경기조사 | 13,338 | 1 | C1:9, C2:16 | C1:업종코드별, C2:BSI코드별 | — | 열 없음 | M:114 | M 200302–201207 | — |
| DT_512Y019 | 지역별 기업경기조사(실적(2013.1~)) | 기업경기조사 | 74,400 | 1 | C1:22, C2:21 | C1:BSI코드별, C2:담당부서별 | — | 열 없음 | M:164 | M 201301–202608 | 2026-09-20 |
| DT_512Y020 | 지역별 기업경기조사(전망(2013.2~)) | 기업경기조사 | 74,400 | 1 | C1:22, C2:21 | C1:BSI코드별, C2:담당부서별 | — | 열 없음 | M:164 | M 201302–202609 | 2026-09-20 |
| DT_512Y021 | 경영애로사항 | 기업경기조사 | 7,876 | 1 | C1:2, C2:14 | C1:업종코드별, C2:경영애로사항코드별 | 1 | % | M:284 | M 200301–202608 | 2026-09-20 |
| DT_513Y001 | 경제심리지수 | 경제심리지수 | 568 | 1 | C1:2 | C1:계정코드별 | — | 열 없음 | M:284 | M 200301–202608 | 2026-09-20 |
| DT_514Y001 | 대출행태서베이(대출태도) | 금융기관대출행태조사 | 701 | 1 | C1:9 | C1:계정코드별 | — | 열 없음 | Q:99 | Q 200201–202603 | 2026-09-20 |
| DT_514Y002 | 대출행태서베이(신용위험) | 금융기관대출행태조사 | 602 | 1 | C1:8 | C1:계정코드별 | — | 열 없음 | Q:99 | Q 200201–202603 | 2026-09-20 |
| DT_514Y003 | 대출행태서베이(대출수요) | 금융기관대출행태조사 | 701 | 1 | C1:9 | C1:계정코드별 | — | 열 없음 | Q:99 | Q 200201–202603 | 2026-09-20 |
| DT_601Y003 | 신용카드 | 지급결제통계 | 12,690 | 1 | C1:15, C2:3 | C1:계정항목별, C2:금융기관 유형코드별 | 3 | 백만원, 천건, 천장 | M:282 | M 200301–202606 | 2026-09-20 |
| DT_602Y001 | 결제시스템별 통계 | 지급결제통계 | 15,018 | 1 | C1:20, C2:2 | C1:계정항목별, C2:금액구분코드별 | 2 | 십억원, 천건 | M:427 | M 199101–202607 | 2026-09-20 |
| DT_602Y002 | 지급수단별 통계 | 지급결제통계 | 19,508 | 1 | C1:27, C2:2 | C1:계정항목별, C2:금액구분코드별 | 2 | 십억원, 천건 | M:426 | M 199101–202606 | 2026-09-20 |
| DT_603Y001 | 한은금융망결제통계표 | 지급결제통계 | 16,558 | 1 | C1:30, C2:2 | C1:계정항목별, C2:구분코드별 | 2 | 건, 백만원 | M:284 | M 200301–202608 | 2026-09-20 |
| DT_603Y010 | 지급결제동향 | 지급결제통계 | 24,628 | 1 | C1:41, C2:2 | C1:계정항목별, C2:구분코드별 | 2 | 건, 백만원 | M:320 | M 200001–202608 | 2026-09-20 |
| DT_604Y003 | 지로시스템 | 지급결제통계 | 3,512 | 1 | C1:4, C2:2 | C1:계정항목별, C2:금액구분코드별 | 2 | 십억원, 천건 | M:439 | M 199001–202607 | 2026-09-20 |
| DT_605Y012 | 금융공동망 | 지급결제통계 | 4,816 | 1 | C1:7, C2:2 | C1:계정항목별, C2:금액구분코드별 | 2 | 십억원, 천건 | M:439 | M 199001–202607 | 2026-09-20 |
| DT_606Y002 | 한은금융망 | 지급결제통계 | 6,080 | 1 | C1:8, C2:2 | C1:계정항목별, C2:금액구분코드별 | 2 | 건, 십억원 | M:380 | M 199501–202608 | 2026-09-20 |
| DT_633Y002 | 전자지급결제대행 | 지급결제통계 | 684 | 1 | C1:2, C2:5 | C1:구분코드별, C2:계정항목별 | 2 | 억원, 천건 | Q:74 | Q 200801–202602 | 2026-09-20 |
| DT_633Y003 | 결제대금예치 | 지급결제통계 | 148 | 1 | C1:2 | C1:계정항목별 | 2 | 억원, 천건 | Q:74 | Q 200801–202602 | 2026-09-20 |
| DT_633Y004 | 선불전자지급수단 | 지급결제통계 | 148 | 1 | C1:2 | C1:계정항목별 | 2 | 억원, 천건 | Q:74 | Q 200801–202602 | 2026-09-20 |
| DT_633Y005 | 전자고지결제 | 지급결제통계 | 148 | 1 | C1:2 | C1:계정항목별 | 2 | 억원, 천건 | Q:74 | Q 200801–202602 | 2026-09-20 |
| DT_633Y006 | 직불전자지급수단 | 지급결제통계 | 108 | 1 | C1:2 | C1:계정항목별 | 2 | 건, 백만원 | Q:54 | Q 201301–202602 | 2026-09-20 |
| DT_641Y001 | 한국은행 원화대출금 | 통화금융통계 | 5,369 | 1 | C1:12 | C1:계정항목별 | 1 | 십억원 | M:668 | M 197101–202608 | 2026-09-20 |
| DT_801Y002 | 어음교환 및 부도 | 지급결제통계 | 8,093 | 1 | C1:12 | C1:계정항목별 | 5 | %, 개, 십억원, 천원, 천장 | M:799 | M 196001–202607 | 2026-09-20 |
| DT_801Y003 | 지역별 어음부도율 | 지급결제통계 | 6,844 | 1 | C1:21 | C1:계정항목별 | 1 | % | M:355 | M 199701–202607 | 2026-09-20 |

In [6]:
print("행 수가 가장 많은 표 15개:")
show_table(["표 ID", "표 이름", "행 수", "항목 수", "분류 차원 수", "주기"], [
    (s["table_id"], names.get(s["table_id"], "카탈로그 없음"),
     f'{s["rows"]:,}', s["item_count"], len(s["dimensions"]),
     ", ".join(sorted(s["period_counts"])))
    for s in sorted(table_summary, key=lambda s: s["rows"], reverse=True)[:15]
])
unit_tables = Counter(unit for s in table_summary for unit in s["unit_values"])
print("단위별 표 수 (상위 20개):")
show_table(["단위", "표 수"], unit_tables.most_common(20))


행 수가 가장 많은 표 15개:


| 표 ID | 표 이름 | 행 수 | 항목 수 | 분류 차원 수 | 주기 |
|---|---|---|---|---|---|
| DT_511Y002 | 소비자동향조사(전국, 월, 2008.9~) | 117,698 | 1 | 2 | M |
| DT_512Y019 | 지역별 기업경기조사(실적(2013.1~)) | 74,400 | 1 | 2 | M |
| DT_512Y020 | 지역별 기업경기조사(전망(2013.2~)) | 74,400 | 1 | 2 | M |
| DT_511Y004 | 소비자동향조사(지역, 월) | 68,282 | 1 | 2 | M |
| DT_303Y004 | 지역별 결제통화(수입) | 58,572 | 1 | 2 | M |
| DT_303Y001 | 지역별 결제통화(수출) | 58,332 | 1 | 2 | M |
| DT_303Y002 | 국가별 결제통화(수출) | 42,684 | 1 | 2 | M |
| DT_303Y005 | 국가별 결제통화(수입) | 42,208 | 1 | 2 | M |
| DT_141Y007 | 비은행금융기관 지역별 여신(말잔) | 36,954 | 1 | 2 | M |
| DT_301Y014 | 서비스무역세분류통계 | 36,062 | 1 | 1 | M |
| DT_141Y010 | 비은행금융기관 지역별 기업대출(말잔) | 34,992 | 1 | 3 | M |
| DT_151Y006 | 비은행금융기관 지역별 가계대출(말잔) | 31,656 | 1 | 2 | M |
| DT_200Y152 | 일반정부의 부문별 기능별 총지출 (명목, 연간) | 31,460 | 1 | 2 | A |
| DT_405Y006 | 국내공급물가지수 | 31,002 | 1 | 1 | M |
| DT_200Y151 | 일반정부의 부문별 기능별 최종소비지출(명목, 연간) | 26,620 | 1 | 2 | A |

단위별 표 수 (상위 20개):


| 단위 | 표 수 |
|---|---|
| 십억원 | 162 |
| % | 45 |
| 백만달러 | 24 |
| 백만원 | 22 |
| 2020＝100 | 21 |
| 십만원 | 12 |
| 천건 | 9 |
| 연리% | 6 |
| %p | 4 |
| 회 | 4 |
| 건 | 4 |
| 억원 | 4 |
| 연% | 2 |
| 천장 | 2 |
| 회/월 | 1 |
| 달러 | 1 |
| 만원 | 1 |
| 억달러 | 1 |
| 2022.12＝100 | 1 |
| 개 | 1 |

## 값·키·메타정보 점검

`DT`의 숫자 변환 가능 여부, 조회 키 중복, 단위 열의 부재와 빈값을 구분합니다. 이 결과는 **점검 후보**이며 KOSIS 원본 정의를 확인하기 전에는 오류로 확정하지 않습니다.


In [7]:
checks = [
    ("CSV 미수집 표", len(missing_ids)),
    ("빈 CSV", sum(s["rows"] == 0 for s in table_summary)),
    ("파일명과 TBL_ID가 다른 행", sum(s["id_mismatch"] for s in table_summary)),
    ("동일 조회 키 중복 행", sum(s["duplicate_keys"] for s in table_summary)),
    ("ITM_ID가 빈 행", sum(s["blank_item"] for s in table_summary)),
    ("주기 또는 수록 시점이 빈 행", sum(s["blank_period"] for s in table_summary)),
    ("DT 값이 빈 행", sum(s["blank_value"] for s in table_summary)),
    ("DT 값이 숫자로 변환되지 않는 행", sum(s["non_numeric"] for s in table_summary)),
    ("최종 수정일이 빈 행", sum(s["blank_update"] for s in table_summary)),
    ("UNIT_NM 열이 없는 CSV", sum(not s["has_unit_column"] for s in table_summary)),
    ("UNIT_NM 열은 있으나 값이 빈 행", sum(s["blank_unit"] for s in table_summary)),
]
print("점검 결과:")
show_table(["점검 항목", "건수"], [(label, f"{count:,}") for label, count in checks])
print("숫자가 아닌 DT 값 예시:", non_numeric_examples.most_common(20))
print("분류 코드 열별 빈값 행 수:")
dimension_blanks = Counter()
for s in table_summary:
    dimension_blanks.update(s["blank_dimensions"])
show_table(["분류 코드 열", "빈값 행 수"], sorted(dimension_blanks.items()))

flagged = [s for s in table_summary if any(s[key] for key in
           ("id_mismatch", "duplicate_keys", "blank_item", "blank_period",
            "blank_value", "non_numeric", "blank_unit"))]
print("점검 후보 표 상위 20개:")
show_table(["표 ID", "행 수", "중복 키", "항목 빈값", "시점 빈값",
            "DT 빈값", "DT 비숫자", "단위 빈값"], [
    (s["table_id"], f'{s["rows"]:,}', s["duplicate_keys"], s["blank_item"],
     s["blank_period"], s["blank_value"], s["non_numeric"], s["blank_unit"])
    for s in sorted(flagged, key=lambda s: sum(s[key] for key in
                    ("id_mismatch", "duplicate_keys", "blank_item", "blank_period",
                     "blank_value", "non_numeric", "blank_unit")), reverse=True)[:20]
])


점검 결과:


| 점검 항목 | 건수 |
|---|---|
| CSV 미수집 표 | 2 |
| 빈 CSV | 0 |
| 파일명과 TBL_ID가 다른 행 | 0 |
| 동일 조회 키 중복 행 | 0 |
| ITM_ID가 빈 행 | 0 |
| 주기 또는 수록 시점이 빈 행 | 0 |
| DT 값이 빈 행 | 0 |
| DT 값이 숫자로 변환되지 않는 행 | 0 |
| 최종 수정일이 빈 행 | 339,040 |
| UNIT_NM 열이 없는 CSV | 53 |
| UNIT_NM 열은 있으나 값이 빈 행 | 0 |

숫자가 아닌 DT 값 예시: []
분류 코드 열별 빈값 행 수:


| 분류 코드 열 | 빈값 행 수 |
|---|---|

점검 후보 표 상위 20개:


| 표 ID | 행 수 | 중복 키 | 항목 빈값 | 시점 빈값 | DT 빈값 | DT 비숫자 | 단위 빈값 |
|---|---|---|---|---|---|---|---|

## 검색·역질문용 표 후보

표 이름의 `원계열/계절조정`, `말잔/평잔`, `실적/전망` 표현만 통일해 비슷한 이름을 묶습니다. **같은 지표라는 판정이나 골든셋 정답이 아닙니다.** 각 표의 항목·단위·주기를 확인한 뒤 사례로 채택하세요.


In [8]:
def normalize_name(name):
    name = re.sub(r"계절조정계열|계절조정|원계열", "{계열}", name)
    name = re.sub(r"말잔|평잔", "{잔액기준}", name)
    return re.sub(r"실적|전망", "{조사시점}", name)

candidate_groups = defaultdict(list)
for table in tables:
    normalized = normalize_name(table["tbl_nm"])
    if "{" in normalized:
        candidate_groups[(categories[table["tbl_id"]], normalized)].append(table)
candidate_groups = [(category, label, group)
                    for (category, label), group in candidate_groups.items()
                    if len({t["tbl_nm"] for t in group}) >= 2]
print("표 이름 변형 후보 그룹:", len(candidate_groups))
show_table(["분류", "공통 이름", "표 ID와 원래 이름"], [
    (category, label, "; ".join(f'{t["tbl_id"]} {t["tbl_nm"]}' for t in group))
    for category, label, group in sorted(candidate_groups,
                                         key=lambda item: (-len(item[2]), item[0], item[1]))[:30]
])


표 이름 변형 후보 그룹: 15


| 분류 | 공통 이름 | 표 ID와 원래 이름 |
|---|---|---|
| 통화금융통계 | M1 상품별 구성내역({잔액기준}, {계열}) | DT_101Y016 M1 상품별 구성내역(말잔, 계절조정계열); DT_101Y017 M1 상품별 구성내역(말잔, 원계열); DT_101Y018 M1 상품별 구성내역(평잔, 계절조정계열); DT_101Y019 M1 상품별 구성내역(평잔, 원계열); DT_161Y001 M1 상품별 구성내역(평잔, 계절조정계열); DT_161Y002 M1 상품별 구성내역(평잔, 원계열); DT_161Y003 M1 상품별 구성내역(말잔, 계절조정계열); DT_161Y004 M1 상품별 구성내역(말잔, 원계열) |
| 통화금융통계 | M2 경제주체별 보유현황({잔액기준}, {계열}) | DT_101Y005 M2 경제주체별 보유현황(말잔, 계절조정계열); DT_101Y006 M2 경제주체별 보유현황(평잔, 계절조정계열); DT_101Y014 M2 경제주체별 보유현황(말잔, 원계열); DT_101Y015 M2 경제주체별 보유현황(평잔, 원계열); DT_161Y009 M2 경제주체별 보유현황(평잔, 계절조정계열); DT_161Y010 M2 경제주체별 보유현황(평잔, 원계열); DT_161Y011 M2 경제주체별 보유현황(말잔, 계절조정계열); DT_161Y012 M2 경제주체별 보유현황(말잔, 원계열) |
| 통화금융통계 | M2 상품별 구성내역({잔액기준}, {계열}) | DT_101Y001 M2 상품별 구성내역(말잔, 계절조정계열); DT_101Y002 M2 상품별 구성내역(말잔, 원계열); DT_101Y003 M2 상품별 구성내역(평잔, 계절조정계열); DT_101Y004 M2 상품별 구성내역(평잔, 원계열); DT_161Y005 M2 상품별 구성내역(평잔, 계절조정계열); DT_161Y006 M2 상품별 구성내역(평잔, 원계열); DT_161Y007 M2 상품별 구성내역(말잔, 계절조정계열); DT_161Y008 M2 상품별 구성내역(말잔, 원계열) |
| 통화금융통계 | 본원통화 구성내역({잔액기준}, {계열}) | DT_102Y001 본원통화 구성내역(말잔, 원계열); DT_102Y002 본원통화 구성내역(평잔, 원계열); DT_102Y003 본원통화 구성내역(말잔, 계절조정계열); DT_102Y004 본원통화 구성내역(평잔, 계절조정계열); DT_102Y101 본원통화 구성내역(말잔, 원계열); DT_102Y102 본원통화 구성내역(평잔, 원계열); DT_102Y103 본원통화 구성내역(말잔, 계절조정계열); DT_102Y104 본원통화 구성내역(평잔, 계절조정계열) |
| 통화금융통계 | Lf 상품별 구성내역({잔액기준}, {계열}) | DT_111Y001 Lf 상품별 구성내역(말잔, 계절조정계열); DT_111Y002 Lf 상품별 구성내역(말잔, 원계열); DT_111Y003 Lf 상품별 구성내역(평잔, 계절조정계열); DT_111Y004 Lf 상품별 구성내역(평잔, 원계열); DT_171Y001 Lf 상품별 구성내역(말잔, 계절조정계열); DT_171Y002 Lf 상품별 구성내역(말잔, 원계열) |
| 통화금융통계 | L(광의유동성) 구성내역({잔액기준}, {계열}) | DT_112Y001 L(광의유동성) 구성내역(말잔, 계절조정계열); DT_112Y002 L(광의유동성) 구성내역(말잔, 원계열); DT_172Y001 L(광의유동성) 구성내역(말잔, 계절조정계열); DT_172Y002 L(광의유동성) 구성내역(말잔, 원계열) |
| 기업경기조사 | 기업경기조사({조사시점}) | DT_512Y013 기업경기조사(실적); DT_512Y014 기업경기조사(전망) |
| 기업경기조사 | 기업경기조사(매출액가중 {조사시점}) | DT_512Y015 기업경기조사(매출액가중 실적); DT_512Y016 기업경기조사(매출액가중 전망) |
| 기업경기조사 | 업종별 기업경기실사지수, ~2009.8({조사시점}) | DT_512Y001 업종별 기업경기실사지수, ~2009.8(실적); DT_512Y002 업종별 기업경기실사지수, ~2009.8(전망) |
| 통화금융통계 | Lf 구성내역({잔액기준}, {계열}) | DT_171Y003 Lf 구성내역(평잔, 계절조정계열); DT_171Y004 Lf 구성내역(평잔, 원계열) |
| 통화금융통계 | M2 기관별 구성내역({잔액기준}) | DT_161Y013 M2 기관별 구성내역(평잔); DT_161Y014 M2 기관별 구성내역(말잔) |
| 통화금융통계 | M2 기관별구성내역({잔액기준}) | DT_101Y007 M2 기관별구성내역(말잔); DT_101Y008 M2 기관별구성내역(평잔) |
| 통화금융통계 | 비은행금융기관 수신({잔액기준}) | DT_111Y007 비은행금융기관 수신(말잔); DT_111Y008 비은행금융기관 수신(평잔) |
| 통화금융통계 | 예금은행 원화예금(종별, {잔액기준}) | DT_104Y008 예금은행 원화예금(종별, 평잔); DT_104Y015 예금은행 원화예금(종별, 말잔) |
| 통화금융통계 | 예금은행 총수신({잔액기준}) | DT_104Y013 예금은행 총수신(말잔); DT_104Y014 예금은행 총수신(평잔) |